# GNSS Timeseries Project

## Documentation overview

## TODO Summary

- format tooltip (pop-ups are slow) #COMPLETE#
- fix vector scaling (and check for correctness)
- fix vector reference bar
- fix vector coloring (up displacement)
- error ellipses
- look into projections (polar, etc.)
- add support for additional organizations/data sets
- add support for manually selecting data folder storage directory
- implement Docker package management
- Fix case sensitivity on everywhere (especially site lookup)


This notebook reads and displays velocity fields and time series from the NGF (EarthScope data center), UNR (University Nevada Reno), and JPL (Jet Propulsion Laboratory).  Most likely additional packages such as ipyleaflet, pandas, numpy, ipywidgets, matplotlib, geopy, json will need to be installed.  Conda installations from conda-forge should be possible for all packages.

The widget interface at the bottom of the notebook has three cells:

<b>Availability:</b> The “Latest NGF”, “Latest UNR”, and “Latest JPL” buttons must be used the first time the Notebook is run to create the data directory and subdirectories for each center that will used to save JSON files with information about all sites at the centers and time series files from each center as they are requested.  Latest updates should be selected to ensure the most up-to-date information is available.  The Availability check can used to see what is known about a 4-char code site name at each of the centers.

<b>Map:</b> The Map interface allows locations of sites, along with other sites within a user-specified radius to be plotted, along with 3-D velocities if available and selected.  Distance table for the 10 nearest sites to a site can be generated.

<b>Timeseries:</b> The Time series interface allows time series to be detrended, with optionally breaks from the EarthScope database removed.  Time series from different sites and centers can be overlayed.  Different backends can be used for plotting time series, with the ipympl backend allowing zooming and saving and the interactive one allowing identification of points.  The default inline backend should work on all systems.  The other backends may not work on all systems. 

All of the cells in the Notebook can be run, and the widgets and additional documentation will be at the bottom of the Notebook.  The code should run in a Jupyter Notebook or VSCode.

## Internal Code


### Importing Packages

In [237]:
import sys
import ast
from pathlib import Path

# Note: Added to keep shared folders working when this notebook is run from inside Main Project.
PROJECT_ROOT = Path.cwd().resolve()
if (PROJECT_ROOT.parent / "data" / "NGF" / "NGF_data.json").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"

local_ipyleaflet_packages = PROJECT_ROOT / "ipyleaflet_packages"
if local_ipyleaflet_packages.exists():
    sys.path.insert(0, str(local_ipyleaflet_packages))

try:
    import ipyleaflet
except:
    print('ipyleaflet not found in virtual environment. Installing...')
    %pip install ipyleaflet
    import ipyleaflet
    print('ipyleaflet installed in virtual environment')

from ipyleaflet import (
    Map, CircleMarker, Circle, Polyline, Marker, DivIcon, Popup,
    TileLayer, WidgetControl, LayerGroup, basemaps
)

import pandas as pd
import numpy as np
import ipywidgets as wg
from traitlets import Unicode
from ipywidgets import HBox, VBox, HTML, Layout
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import gridspec
import math
try:
    import geopy.distance
except:
    print("geopy not found in virtual environment. Installing...")
    %pip install geopy
    import geopy.distance
    print("geopy installed in virtual environment")
import json
import html as html_module
import os
from os.path import exists
from os import makedirs
from datetime import datetime, timedelta, timezone
import urllib.request
import requests
import re
import importlib
import gnss_core
importlib.reload(gnss_core)
from gnss_core import (
    build_map_render_plan, detrended as core_detrended, filter_sigmas as core_filter_sigmas,
    fit_ts as core_fit_ts, number_list as core_number_list, parse_bulk_selection as core_parse_bulk_selection,
    parse_map_selection, remove_brac as core_remove_brac, vec_add as core_vec_add,
    vec_sub as core_vec_sub, velocity_endpoint as core_velocity_endpoint,
    VELOCITY_REFERENCE_ZOOM, VELOCITY_REFERENCE_LATITUDE, format_display_number, format_neu_vector, neighbor_velocity_row,
    velocity_arrow_geometry, velocity_guide_metrics, resolve_popup_owner,
    absolute_vector_render_spec, clear_absolute_vector_render_cache,
    relative_vector_render_spec,
)
from gnss_map_runtime import stage_layer_update
import textwrap
import webbrowser
import ipympl
from IPython import get_ipython
from IPython.display import Javascript

class GNSSDivIcon(DivIcon):
    """DivIcon with Leaflet's custom class_name option across ipyleaflet versions."""
    class_name = Unicode("").tag(sync=True, o=True)

def _vector_div_icon(html: str, icon_size: tuple, icon_anchor: tuple, class_name: str) -> DivIcon:
    icon = GNSSDivIcon(html=html, icon_size=icon_size, icon_anchor=icon_anchor, class_name=class_name)
    return icon

VECTOR_PANE_NAME = 'gnss-vector-pane'
STATION_PANE_NAME = 'gnss-station-pane'
GNSS_MAP_PANES = {
    VECTOR_PANE_NAME: {'zIndex': 410, 'pointerEvents': 'none'},
    STATION_PANE_NAME: {'zIndex': 620, 'pointerEvents': 'auto'},
}

try:
    import earthscope_sdk
except:
    print("earthscope_sdk not found in virtual environment. Installing...")
    %pip install earthscope_sdk
    import earthscope_sdk
    print("earthscope_sdk installed in virtual environment")
esver = earthscope_sdk.__version__

print("\nPackages loaded successfully \nRunning on the following versions:")
print(f"python: {sys.version.split()[0]}")
print(f"ipyleaflet: {ipyleaflet.__version__}")
print(f"ipympl: {ipympl.__version__}")
print(f"earthscope_sdk: {esver}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"ipywidgets: {wg.__version__}")
print(f"matplotlib: {mpl.__version__}")
print(f"requests: {requests.__version__}")


Packages loaded successfully 
Running on the following versions:
python: 3.11.15
ipyleaflet: 0.20.0
ipympl: 0.9.8
earthscope_sdk: 1.5.0
pandas: 3.0.2
numpy: 2.4.6
ipywidgets: 8.1.7
matplotlib: 3.10.9
requests: 2.34.2


### EarthScope Verification

In [238]:
## TODO: EarthScope auth now opens the login/data portal when needed
# Define Earthscope access functions and then see if a test download is to be
# executed. Once access has been tested, the NGF_test variable can be set to False
# to avoid repeated testing and login prompts.
NGF_test = False
# Functions for accessing Earthscope das set that depend on the which version of earthscpoe_sdk
# is installled. These functions must be defined even if the NGF access is not being tested.
if (esver[0] == '0') :

    from earthscope_sdk.auth.device_code_flow import DeviceCodeFlowSimple
    from earthscope_sdk.auth.auth_flow import NoTokensError

    def get_es_file(url: str, directory_to_save_file: object = None, token_path: str = './') -> None:
        """Downloads a file from gage-data.earthscope.org using the EarthScope SDK.

        Args:
            url: url of desired file at gage-data.earthscope.org
            directory_to_save_file: path of directory in which to save the file
            token_path: path of directory in which to save the token

        Raises:
            NoTokensError: if no token is found and the device code flow fails
        """

        if directory_to_save_file is None:
            directory_to_save_file = DATA_DIR / "NGF"

        device_flow = DeviceCodeFlowSimple(Path(token_path))

        try:
            device_flow.get_access_token_refresh_if_necessary()
        except NoTokensError:
            device_flow.do_flow()
        token = device_flow.access_token

        file_name = Path(url).name
        r = requests.get(url, headers={"authorization": f"Bearer {token}"})

        # Have to check the status code because the EarthScope SDK does not raise an exception for a failed request
        if r.status_code == requests.codes.ok:
            with open(Path(Path(directory_to_save_file) / file_name), 'wb') as f:
                for data in r:
                    f.write(data)
        else:
            print(f"failure: {r.status_code}, {r.reason}")

elif (esver[0] == '1') :

    # These imports are needed for the EarthScope SDK >= 1.0.0, which uses OAuth2 authentication
    import time
    from earthscope_sdk import EarthScopeClient

    BASE_URL = "https://data.earthscope.org/archive/gnss/products/position"

    def build_csv_url(station: str, author: str = "cwu", frame: str = "igs14") -> str:
        """Builds the URL for a non-detrended GNSS position time-series CSV.

        Args:
            station: the station code in the EarthScope archive
            author: author code for the GNSS product
            frame: reference frame for the GNSS product

        Returns:
            A URL string for the CSV file

        Raises:
            ValueError: if the station code is empty or invalid

        Example:
            https://data.earthscope.org/archive/gnss/products/position/P162/P162.cwu.igs14.csv
        """
        try:
            st = station.strip().upper()
            if not st:
                raise ValueError
            return f"{BASE_URL}/{st}/{st}.{author.lower()}.{frame.lower()}.csv"
        except Exception as e:
            raise ValueError(f"Invalid station code: {station}") from e


    def fetch_csv(url: str, outpath: Path, headers: dict = None, retries: int = 3, backoff: float = 1.5) -> tuple:
        """Downloads a CSV file with repeated attempts and basic response checking.

        Args:
            url: URL of the CSV file to download
            outpath: local path where the CSV file will be saved
            headers: optional HTTP headers to include in the request
            retries: total number of download attempts
            backoff: delay multiplier between attempts, in seconds

        Returns:
            A tuple containing the URL, whether the download succeeded, and an error message.
            If the download succeeds, the error message is an empty string.
        """
        err = ""
        for attempt in range(1, retries + 1):
            try:
                r = requests.get(url, headers=headers, timeout=30)
                if r.ok:
                    outpath.parent.mkdir(parents=True, exist_ok=True)
                    outpath.write_bytes(r.content)
                    return (url, True, "")
                err = f"HTTP {r.status_code} {r.reason}"
            except requests.RequestException as e:
                err = str(e)
            time.sleep(backoff ** attempt)
        return (url, False, err)

    def download_es_file(es_client: EarthScopeClient, station: str, frame: str = "igs14", outdir: object = None) -> None:
        """Downloads a non-detrended GNSS position time-series CSV from EarthScope.

        Args:
            es_client: authenticated EarthScope client
            station: station code to download
            frame: reference frame for the GNSS product
            outdir: local directory where the CSV file will be saved
        """

        if outdir is None:
            outdir = DATA_DIR / "NGF"

        outdir = Path(outdir)
        outdir.mkdir(parents=True, exist_ok=True)

        token = es_client.ctx.auth_flow.access_token
        headers = {"Authorization": f"Bearer {token}"}

        failures = []
        url = build_csv_url(station, "cwu", frame)
        fname = Path(url).name
        outpath = outdir / fname
        url, ok, err = fetch_csv(url, outpath, headers=headers)
        if ok:
            print(f"✓ {fname}")
        else:
            print(f"✗ {fname}  --  {err}")
            failures.append((fname, err))

        if failures:
            print("\n⚠️ Some downloads failed — check author/station/frame availability.")

    print('New EarthScope_SDK version',esver,'set up')

else:
    print('Unknown earthscope_sdk version ',esver)

# Test URL to try to download a file from EarthScope; may require login, depending on EarthScope_sdk version
url = "https://gage-data.earthscope.org/archive/gnss/products/velocity/cwu.final_igs14.vel"
filename = "cwu.final_igs14.vel"

if ( NGF_test and esver[0]== '0' ) :
    print("Getting ",url)
    print("Link for login may appear below")
    get_es_file(url)

if ( NGF_test and esver[0] == '1' ):
    try:
        from earthscope_sdk import EarthScopeClient
        from earthscope_sdk.auth.auth_flow import NoAccessTokenError
        es_client = EarthScopeClient()
        token = es_client.ctx.auth_flow.access_token
        headers = {"Authorization": f"Bearer {token}"}
    except NoAccessTokenError:
        print("EarthScope authentication required. Opening the EarthScope data portal...")
        webbrowser.open("https://data.earthscope.org")
        print("If the browser login does not create a token, run `es login` in your terminal, then re-run this cell.")
        raise SystemExit("EarthScope login needed; run es login and try again.")

    outpath = Path("./"+filename)
    print('Calling fetch_csv with ',url,filename)
    url, ok, err = fetch_csv(url, outpath, headers=headers)
    if ok:
        print(f"✓ {filename}")
    else:
        print(f"✗ {filename}  --  {err}")


New EarthScope_SDK version 1.5.0 set up


### Setting up Directories

In [ ]:
# "LOC" is reserved for user-supplied local station files; it is never downloaded.
orglist = ["NGF", "UNR", "JPL", "LOC"]

GNSS_CONFIG_PATH = Path.home() / ".gnss_analysis.json"
GNSS_DATA_ROOT_ENV = "GNSS_DATA_ROOT"


def make_if_absent(folderpath: str) -> None:
    """Creates a folder for storing data if it does not already exist.

    Args:
        folderpath: path of the folder to check or create
    """
    if not exists(folderpath): makedirs(folderpath)


def data_root_is_writable(path: object) -> bool:
    """Reports whether a directory can actually be written to.

    Creating the folder is not proof it is usable, so this writes and removes a
    probe file.

    Args:
        path: candidate directory.

    Returns:
        True when a file can be created inside the directory.
    """
    try:
        candidate = Path(path)
        candidate.mkdir(parents=True, exist_ok=True)
        probe = candidate / ".gnss_write_probe"
        probe.write_text("ok")
        probe.unlink()
        return True
    except Exception:
        return False


def save_data_root(path: object) -> None:
    """Remembers the chosen data directory between sessions.

    Args:
        path: directory to store as the preferred data root.
    """
    try:
        GNSS_CONFIG_PATH.write_text(json.dumps({"data_root": str(path)}, indent=1))
    except Exception as config_error:
        print("Could not save the data-directory preference:", config_error)


def load_saved_data_root() -> object:
    """Returns the remembered data directory, or None when there is not one."""
    try:
        if GNSS_CONFIG_PATH.exists():
            stored = json.loads(GNSS_CONFIG_PATH.read_text()).get("data_root")
            if stored:
                return Path(stored)
    except Exception:
        pass
    return None


def resolve_initial_data_root(fallback: object) -> object:
    """Chooses the data directory at startup.

    Order: the GNSS_DATA_ROOT environment variable, then the remembered choice,
    then whatever the notebook worked out from its own location.

    Args:
        fallback: directory to use when nothing else is configured.

    Returns:
        The directory to read and write GNSS data in.
    """
    from os import environ
    env_value = environ.get(GNSS_DATA_ROOT_ENV)
    if env_value and data_root_is_writable(env_value):
        return Path(env_value)
    saved = load_saved_data_root()
    if saved is not None and data_root_is_writable(saved):
        return saved
    return Path(fallback)


def load_data_cache() -> None:
    """Reloads data_of from whatever DATA_DIR currently points at.

    Called at startup and again whenever the data directory changes, so switching
    folders does not require re-downloading anything already cached there.
    """
    global data_of
    data_of = {}
    for org in orglist:
        filepath = DATA_DIR / org / f"{org}_data.json"
        if exists(filepath):
            with open(filepath, "r") as data_file:
                data_of[org] = json.load(data_file)


def apply_data_root(path: object, remember: bool = True) -> bool:
    """Points the notebook at a different data directory.

    Rebinding the DATA_DIR global is what re-targets the readers and writers,
    because they look it up at call time. The subdirectories and the in-memory
    cache are rebuilt here since those were resolved once at startup.

    Args:
        path: directory to use.
        remember: whether to store the choice for future sessions.

    Returns:
        True when the directory was accepted.
    """
    global DATA_DIR
    if not data_root_is_writable(path):
        return False
    DATA_DIR = Path(path)
    make_if_absent(DATA_DIR)
    for orgname in orglist:
        make_if_absent(DATA_DIR / orgname)
    load_data_cache()
    if remember:
        save_data_root(DATA_DIR)
    return True


DATA_DIR = resolve_initial_data_root(DATA_DIR)
make_if_absent(DATA_DIR)
for orgname in orglist:
    make_if_absent(DATA_DIR / orgname)


### Reading from Existing Data

In [ ]:
data_of = {}
load_data_cache()


### General Functions

In [ ]:
def number_list(text: str, default: list) -> list:
    """Reads a list or tuple of numbers entered in a widget.

    Args:
        text: text such as "(10, 10, 30)"
        default: values to return when the text is not a valid numeric list

    Returns:
        The entered values as floats, or a copy of the default values.
    """
    return core_number_list(text, default)


def filter_sigmas(df: pd.DataFrame, columns: list, limits: tuple, conversion: float = 1.0) -> pd.DataFrame:
    """Removes rows whose uncertainty is above an enabled component limit.

    Args:
        df: time-series table
        columns: North, East, and Up sigma column numbers
        limits: North, East, and Up limits; zero disables that component
        conversion: number of limit units in one table unit

    Returns:
        The filtered table.
    """
    return core_filter_sigmas(df, columns, limits, conversion)


def FitTS(xdata: np.ndarray, ydata: np.ndarray, sig: np.ndarray) -> tuple:
    """Fits a weighted linear trend to one time-series component.

    Args:
        xdata: time values in days since 2000-01-01
        ydata: position values for one component
        sig: uncertainty values for each position value

    Returns:
        A tuple containing the polynomial fit estimate, velocity estimate with uncertainty,
        and fit statistics as [WRMS, chi, number_of_data].
    """

    ## Variable Reference Table ##
        # yr: years since 2000/1/1
        # wgh: weights, determined inversely proportional to the uncertainties (sigma^2)
        # ndata: total number of data points
        # A: matrix of the normal equation coefficients, calculated as np.transpose([np.ones(ndata), yr])
        # NormEQ: matrix of the normal equations, calculated
        # Bvec: vector of the normal equations, calculated as (X.T * OmInv) @ ydata.T
        # MCov: covariance matrix of the fit parameters, calculated as the inverse of the normal equations
        # MEst: fit parameters, calculated as MCov @ Bvec
        # Res: residuals, calculated from ydata - X @ MEst


    # Set up to fit linear trend.  In later updates we could
    # create more elaborate models with offsets, periodic and post-
    # seismic componensts.
    yr = xdata/365.25
    wgh = 1./sig**2
    ndata = int(xdata.size)
    A = np.transpose([np.ones(ndata), yr])
    NormEQ = np.matmul(np.transpose(A)*wgh, A)
    Bvec = np.matmul(np.transpose(A)*wgh, np.transpose(ydata))
    MCov = np.linalg.inv(NormEQ)
    MEst = np.matmul(MCov,Bvec)
    Res = ydata-np.matmul(A,MEst)
    chi = np.sqrt(np.dot(np.transpose(Res), Res*wgh) / (ndata-2))
    wrms = np.sqrt(ndata/np.sum(wgh)) * chi
    pfe = np.flip(MEst) # Set reverse order like polyfit

    return pfe, [MEst[1],np.sqrt(MCov[1,1])], [wrms,chi,ndata]

def detrended(xdata: np.ndarray, ydata: np.ndarray, sig: np.ndarray) -> tuple:
    """Removes the weighted linear trend from one time-series component.

    Args:
        xdata: time values in days since 2000-01-01
        ydata: position values for one component
        sig: uncertainty values for each position value

    Returns:
        A tuple containing detrended residuals, velocity estimate, and fit statistics.
    """
    pfe , Vel, Stat = FitTS(xdata, ydata, sig)
    return ydata - (pfe[0] * xdata / 365.25 + pfe[1]), Vel, Stat # Stat = statistics = (WRMS, NRMS, #data)


# Use the reusable pure implementations for all later notebook calls.
FitTS = core_fit_ts
detrended = core_detrended

def remove_outliers_function(nd: np.ndarray, ed: np.ndarray, ud: np.ndarray, ns: np.ndarray, es: np.ndarray, us: np.ndarray, times: np.ndarray, td: np.ndarray, multiplier: float) -> tuple:
    """Removes points whose North, East, or Up value is far from the mean.

    Args:
        nd: North position values
        ed: East position values
        ud: Up position values
        ns: North uncertainty values
        es: East uncertainty values
        us: Up uncertainty values
        times: datetime values used for plotting
        td: time values in days since 2000-01-01
        multiplier: number of standard deviations allowed before removal

    Returns:
        a tuple containing the filtered position, uncertainty, datetime, and time arrays.
    """
    bool_list = [all(abs(lst[i]-np.mean(lst)) <= multiplier*np.std(lst) for lst in (nd, ed, ud)) for i in range(len(nd))]
    nd_new = nd[bool_list]
    ed_new = ed[bool_list]
    ud_new = ud[bool_list]
    ns_new = ns[bool_list]
    es_new = es[bool_list]
    us_new = us[bool_list]
    times_new = times[bool_list]
    td_new = td[bool_list]
    return nd_new, ed_new, ud_new, ns_new, es_new, us_new, times_new, td_new


def calc_distance_earth(start_lat: float, start_lon: float, distance_km: float, direction: str = "east") -> list:
    """Calculates a new latitude and longitude after moving along Earth's surface.

    Args:
        start_lat: starting latitude in decimal degrees
        start_lon: starting longitude in decimal degrees
        distance_km: distance to move in kilometers
        direction: direction to move; one of "east", "west", "north", or "south"

    Returns:
        A list containing the new latitude and longitude.
    """

    r = 6371.0 # radius of Earth
    if direction == "east":
        lat_rad = math.radians(start_lat)
        delta = (distance_km/(r * math.cos(lat_rad))) * (180/math.pi)
        end_lon = start_lon + delta
        end_lat = start_lat

    elif direction == "west":
        lat_rad = math.radians(start_lat)
        delta = (distance_km/(r * math.cos(lat_rad))) * (180/math.pi)
        end_lon = start_lon - delta
        end_lat = start_lat

    elif direction == "north":
        delta = distance_km / r
        lat_rad = math.radians(start_lat)
        new_lat_rad = lat_rad + delta
        end_lat = math.degrees(new_lat_rad)
        end_lon = start_lon

    elif direction == "south":
        delta = distance_km / r
        lat_rad = math.radians(start_lat)
        new_lat_rad = lat_rad - delta
        end_lat = math.degrees(new_lat_rad)
        end_lon = start_lon

    return [end_lat, end_lon]


### Timeseries Readers

In [ ]:
## FETCH TIMESERIES DATA

SigLim = (10,10,30) # standard deviation

# UNR Data Fetcher
def Read_UNR(site: str, update: bool) -> tuple:
    """Reads a UNR station time series from the local cache or UNR server.

    Args:
        site: 4-character station code
        update: whether to download a fresh copy instead of using the local file

    Returns:
        A tuple containing datetime values and a NEU time-series array.
    """
    site = site.strip().upper()
    if bool(update):
        clear_absolute_vector_render_cache()
    # TODO: check if site is down first
    # MOD TAH 241223: Use IGS14 time series.
    file_directory = DATA_DIR / "UNR" / f"{site.upper()}.csv"
    file_name = site.upper()+".tenv3"
    fetch_url="https://geodesy.unr.edu/gps_timeseries/IGS20/tenv3/IGS20/"+file_name

    if exists(file_directory) and not bool(update) :
        print('Reading from file ', file_directory)
        df = pd.read_csv(file_directory, delimiter=',')
    else:
        print('Downloading from ', fetch_url)
        try:
            df = pd.read_csv(fetch_url, delimiter=r"\s+", header=1)
            df.to_csv(file_directory, index=False)
        except:
            df = []
            with timeseries_output:
                display(wg.HTML('''<em style="color:red">UNR Site '''+site.upper()+''' cant be found</em>'''))
    ninit = len(df)

    # Note: UNR stores East, North, Up sigmas in meters.
    df = filter_sigmas(df, [15, 14, 16], SigLim, 1000)
    if any(limit > 0 for limit in SigLim):
        print("Number after >{} {} {} mm NEU sigma removal".format(*SigLim),len(df),"Read",ninit)
    npdat = df.to_numpy()

    t=list(npdat[:,1])
    nd=list((npdat[:,10]-npdat[0,10])*1000) # Implemented to remove first value so it starts at zero
    # Note: the same as NGF. (Problem if times of first data point are different)
    ed=list((npdat[:,8]-npdat[0,8])*1000)
    ud=list((npdat[:,12]-npdat[0,12])*1000)
    ns=list(npdat[:,15]*1000) ; es=list(npdat[:,14]*1000) ; us=list(npdat[:,16]*1000)

    n = 0
    to = []
    td = np.zeros(len(t))
    for v in t:
        to = np.append(to, datetime.strptime(v, '%y%b%d')+timedelta(hours=12))
        dt = to[n] - datetime(2000, 1, 1,0,0)  # dt: time difference from 2000/1/1
        td[n] = dt.total_seconds()/86400.  # td: days from 2000/1/1
        n += 1

    tseries = np.array([td,nd,ns,ed,es,ud,us])
    times = to
    return times, tseries

def Read_JPL(site: str, update: bool) -> tuple:
    """Reads a JPL station time series from the local cache or JPL server.

    Args:
        site: 4-character station code
        update: whether to download a fresh copy instead of using the local file

    Returns:
        A tuple containing datetime values and a NEU time-series array.
    """
    site = site.strip().upper()
    if bool(update):
        clear_absolute_vector_render_cache()
    # TODO: check if site is down first
    csv_file_directory = DATA_DIR / "JPL" / f"{site}.csv"

    fetch_url="https://sideshow.jpl.nasa.gov/pub/JPL_GPS_Timeseries/repro2018a/post/point/"+site+".series"

    if exists(csv_file_directory) and not bool(update):
        print('Reading from file',csv_file_directory)
        df = pd.read_csv(csv_file_directory,delimiter=',')
    else:
        print('Downloading from ',fetch_url)
        try:
            df = pd.read_csv(fetch_url,delimiter=r"\s+")
            df.to_csv(csv_file_directory,index=False)
        except:
            df=[]
            with timeseries_output:
                display(wg.HTML('''<em style="color:red">JPL Site '''+site.upper()+''' cant be found</em>'''))

    ninit = len(df)
    # Note: JPL stores East, North, Up sigmas in meters
    df = filter_sigmas(df, [5, 4, 6], SigLim, 1000)
    if any(limit > 0 for limit in SigLim):
        print("Number after >{} {} {} mm NEU sigma removal".format(*SigLim),len(df),"Read",ninit)

    npdat = df.to_numpy()

    nd=list((npdat[:,2]-npdat[0,2])*1000) # Implemented to remove first value so it starts at zero
    # Note: the same as NGF. (Problem if times of first data point are different)
    ed=list((npdat[:,1]-npdat[0,1])*1000) # 2nd column
    ud=list((npdat[:,3]-npdat[0,3])*1000) # 4th column
    ns=list(npdat[:,5]*1000) ; es=list(npdat[:,4]*1000) ; us=list(npdat[:,6]*1000) # what do any of these mean
    to = np.empty(len(npdat[:,10]), dtype = object)

    for i in range(len(to)):
        each_date = datetime((int(npdat[:,11][i])), int((npdat[:,12][i])), int((npdat[:,13][i])))+timedelta(hours=12)
        to[i] = each_date

    td_counter = 0
    td = np.zeros(len(npdat[:,10]))
    for time_value in npdat[:,10]:
        td[td_counter] = time_value/86400 # td: days from 2000/1/1 (file has seconds from 2000/1/1, 12:00 hr)
        td_counter += 1

    tseries = np.array([td,nd,ns,ed,es,ud,us])
    times = to
    return times, tseries




def Read_NGF(site: str, update: bool) -> tuple:
    """Reads an NGF station time series from the local cache or EarthScope server.

    Args:
        site: 4-character station code
        update: whether to download a fresh copy instead of using the local file

    Returns:
        A tuple containing datetime values and a NEU time-series array.
    """
    global SigLim
    site = site.strip().upper()
    if bool(update):
        clear_absolute_vector_render_cache()
    filen = site.upper()+".cwu.igs14.csv"
    filepath = DATA_DIR / "NGF" / filen
    whurl="https://data.unavco.org/archive/gnss/products/position/"+site.upper()+"/"+filen

    #PBO Station Position Time Series.
    #Format Version, 1.2.0
    #Reference Frame, igs14
    #4-character ID, P177
    #Station name, CoDeTierraCN2008
    #Begin Date, 2008-05-09
    #End Date, 2021-10-06
    #Release Date, 2021-10-07
    #Source file, P177.cwu.igs14.pos
    #offset from source file, 166.01 mm North, -139.45 mm East, -3.62 mm Vertical
    #Reference position, 37.5281683684 North Latitude, -122.4950535711 East Longitude, 71.79172 meters elevation
    #Date, North (mm), East (mm), Vertical (mm), North Std. Deviation (mm), East Std. Deviation (mm), Vertical Std. Deviation (mm), Quality,
    #2008-05-09,0.00, 0.00, 0.00, 2.08, 1.66, 7.84, repro,
    #2008-05-10,0.38, 0.83, 2.66, 2.13, 1.7, 7.99, repro,
    #...
    if exists(filepath) and not bool(update):
        print('Reading from file',filen)
        df = pd.read_csv(filepath,delimiter=',',header=11)
    else:
        if ( esver[0] == '0' ) :
            print('Downloading from ',whurl)
            try:
                get_es_file(whurl)
                df = pd.read_csv(filepath,delimiter=',',header=11)
            except:
                df = []
                with timeseries_output:
                    display(wg.HTML('''<em style="color:red">NGF Site '''+site.upper()+''' cant be found</em>'''))

        else:
            # Uses ES 1.0 instead of ES 0.0.1, which is now obsolete.
            print('Getting',site.upper(),'with EarthSopeClient')
            es = EarthScopeClient()
            # Refreshes access token if necessary; if failed, use the EarthScope CLI again to login: es login
            # Example error: "NoRefreshTokenError: No refresh token was found. Please re-authenticate."
            es.ctx.auth_flow.refresh_if_necessary()
            download_es_file(es, site.upper(), frame="igs14")

        df = pd.read_csv(filepath,delimiter=',',header=11)

    ninit = len(df)
    # Note: NGF stores its sigma columns in millimeters already.
    df = filter_sigmas(df, [4, 5, 6], SigLim)
    if any(limit > 0 for limit in SigLim):
        print("Number after >{} {} {} mm NEU sigma removal".format(*SigLim),len(df),"Read",ninit)
    npdat = df.to_numpy()

    t=list(npdat[:,0]);nd=list(npdat[:,1]); ed=list(npdat[:,2]); ud=list(npdat[:,3]);
    ns=list(npdat[:,4]) ; es=list(npdat[:,5]) ; us=list(npdat[:,6])

    n = 0
    to = []
    td = np.zeros(len(t))
    for v in t:
        # Note: moves time to middle of the day because the data is recorded at 12:00 UTC.
        to = np.append(to,datetime.strptime(v, '%Y-%m-%d')+timedelta(hours=12))
        dt = to[n] - datetime(2000, 1, 1)  # Time difference from 2000/1/1
        td[n] = dt.total_seconds()/86400.  # Days from 2000/1/1
        n += 1

    tseries =  np.array([td,nd,ns,ed,es,ud,us])
    times = to
    return times, tseries

def get_local_cwu_metadata(filepath: object) -> dict:
    """Reads position and velocity metadata out of a local CWU position file.

    The station record is built to the same shape the downloaded catalogs use, so
    a local station behaves like any other everywhere else in the notebook.

    Args:
        filepath: path to a CWU-format .csv position file.

    Returns:
        A station record with location, height, velocity and velsig.
    """
    latitude = longitude = height = None
    with open(filepath, "r") as handle:
        header_lines = [next(handle) for _ in range(11)]
    for line in header_lines:
        if line.startswith("#Reference position") or line.startswith("Reference position"):
            parts = line.split(",")
            latitude = float(parts[1].split()[0])
            longitude = float(parts[2].split()[0])
            height = float(parts[3].split()[0])
    if latitude is None:
        raise ValueError("no '#Reference position' line in the first 11 header lines")

    frame = pd.read_csv(filepath, delimiter=",", header=11)
    values = frame.to_numpy()
    days = np.array([
        (datetime.strptime(str(value), "%Y-%m-%d") - datetime(2000, 1, 1)).total_seconds() / 86400.0
        for value in values[:, 0]
    ], dtype=float)

    record = {"location": [latitude, longitude], "height": height}
    velocity, velsig = [], []
    for north_col, sigma_col in ((1, 4), (2, 5), (3, 6)):
        series = np.array(values[:, north_col], dtype=float)
        sigma = np.array(values[:, sigma_col], dtype=float)
        _fit, rate, _stats = FitTS(days, series, sigma)
        velocity.append(rate[0])
        velsig.append(rate[1])
    record["velocity"] = velocity
    record["velsig"] = velsig
    return record


def Read_LOC(site: str, update: bool) -> tuple:
    """Reads a user-supplied local CWU-format time series.

    Args:
        site: 4-character station code, as registered by the local-file loader.
        update: ignored; a local file is never re-downloaded.

    Returns:
        A tuple containing datetime values and a NEU time-series array.
    """
    global SigLim
    site = site.strip().upper()
    record = data_of.get("LOC", {}).get(site)
    if not record or "filename" not in record:
        with timeseries_output:
            display(wg.HTML(
                '<em style="color:#B42318">' + site + ' is not a registered local '
                'file. Add it under Data Location in the Database Fetcher.</em>'))
        return np.array([]), np.array([[], [], [], [], [], [], []])

    filepath = DATA_DIR / "LOC" / record["filename"]
    print("Reading local file", filepath)
    df = pd.read_csv(filepath, delimiter=",", header=11)
    ninit = len(df)
    df = filter_sigmas(df, (4, 5, 6), SigLim)
    print("Number after >{} {} {} mm NEU sigma removal".format(*SigLim), len(df), "Read", ninit)

    npdat = df.to_numpy()
    n = 0
    td = np.zeros(len(npdat)); to = np.array([])
    nd = np.array(npdat[:, 1], dtype=float); ns = np.array(npdat[:, 4], dtype=float)
    ed = np.array(npdat[:, 2], dtype=float); es = np.array(npdat[:, 5], dtype=float)
    ud = np.array(npdat[:, 3], dtype=float); us = np.array(npdat[:, 6], dtype=float)
    for v in npdat[:, 0]:
        to = np.append(to, datetime.strptime(str(v), '%Y-%m-%d') + timedelta(hours=12))
        td[n] = (to[n] - datetime(2000, 1, 1)).total_seconds() / 86400.
        n += 1
    return to, np.array([td, nd, ns, ed, es, ud, us])


Read = {
        "NGF": Read_NGF,
        "UNR": Read_UNR,
        "JPL": Read_JPL,
        "LOC": Read_LOC
    }

### Map Functions

In [243]:
d_km = lambda coords1, coords2: geopy.distance.geodesic(coords1,coords2).km
d_mi = lambda coords1, coords2: geopy.distance.geodesic(coords1,coords2).miles

def add_map_layer(map_obj: Map, layer: object) -> None:
    map_obj.add(layer)


def plot_locations(map_obj: Map, coordslist: list, tooltiplist: list, color: str = "blue", dotrad: int = 3) -> None:
    """Plots simple dot markers on an ipyleaflet map.

    Args:
        map_obj: ipyleaflet map object
        coordslist: list of latitude and longitude pairs
        tooltiplist: list of text labels for the marker popups
        color: marker color
        dotrad: marker radius in pixels
    """
    for coords, tooltip in zip(coordslist, tooltiplist):
        marker = CircleMarker(
            location=coords,
            pane=STATION_PANE_NAME,
            radius=dotrad,
            color=color,
            weight=0,
            fill_color=color,
            fill_opacity=1,
            opacity=1,
        )
        marker.popup = HTML(value=str(tooltip))
        add_map_layer(map_obj, marker)



def map_station_summary(siteid: str, org: str) -> str:
    """Build a complete, hundredths-precision station summary for a popup."""
    try:
        info = data_of[org][siteid]
        loc = info.get("location", [None, None])
        hgt = format_display_number(info.get("height"))
        vel = format_neu_vector(info.get("velocity"))
        sig = format_neu_vector(info.get("velsig"))
        lat = format_display_number(loc[0] if len(loc) > 0 else None)
        lon = format_display_number(loc[1] if len(loc) > 1 else None)
        return (
            f"Site: {siteid}\n"
            f"Source: {org}\n"
            f"Lat/Lon: {lat}, {lon}\n"
            f"Height: {hgt} m\n"
            f"Velocity NEU (mm/yr): {vel}\n"
            f"Sigma NEU (mm/yr): {sig}"
        )
    except (KeyError, TypeError, IndexError):
        return f"{siteid} ({org})"

def add_map_site_to_timeseries(siteid: str, org: str, plot_now: bool = False) -> None:
    """Adds a station selected from the map to the time-series widget.

    Args:
        siteid: 4-character station code
        org: data source name
        plot_now: whether to immediately plot the selected station
    """
    # Note: this was implemented to let the map talk directly to the time-series widget.
    extendedsiteid = siteid + " (" + org + ")"
    if 'ts_site_form' in globals():
        ts_site_form.value = siteid
    if 'org_ts_select' in globals() and org in org_ts_select.options:
        org_ts_select.value = org
    if 'ts_sites' in globals() and extendedsiteid not in ts_sites.options:
        ts_sites.options = list(ts_sites.options) + [extendedsiteid]
    if 'ts_sites' in globals():
        ts_sites.value = (extendedsiteid,)
    if plot_now and 'list_to_graph' in globals():
        list_to_graph(None)


def station_popup(siteid: str, org: str, tooltip: str) -> VBox:
    """Builds the ipywidgets popup shown when a station marker is clicked.

    Args:
        siteid: 4-character station code
        org: data source name
        tooltip: station text to show in the popup

    Returns:
        A VBox widget containing station text, buttons, and copyable data.
    """
    # Keep controls and metadata inside one responsive, bounded popup body.
    button_layout = wg.Layout(width='49%', min_width='0')
    add_button = wg.Button(description="Add to Timeseries", layout=button_layout)
    plot_button = wg.Button(description="Plot Now", button_style="success", layout=button_layout)
    summary = map_station_summary(siteid, org)
    metadata_rows = []
    for line in summary.splitlines():
        label, separator, value = line.partition(':')
        if not separator:
            label, value = 'Station', line
        metadata_rows.append(
            '<div style="font-weight:600; min-width:0">' + html_module.escape(label.strip()) + '</div>'
            '<div style="min-width:0; overflow-wrap:anywhere; word-break:break-word">' + html_module.escape(value.strip()) + '</div>'
        )
    metadata = HTML(
        value=(
            '<div class="gnss-popup-metadata" style="display:grid; grid-template-columns:minmax(7rem,auto) minmax(0,1fr); '
            'column-gap:8px; row-gap:3px; width:100%; max-width:100%; overflow:hidden; white-space:normal">'
            + ''.join(metadata_rows) + '</div>'
        ),
        layout=wg.Layout(width='100%', max_width='100%', overflow='hidden'),
    )

    def add_clicked(_button):
        add_map_site_to_timeseries(siteid, org, plot_now=False)

    def plot_clicked(_button):
        add_map_site_to_timeseries(siteid, org, plot_now=True)

    add_button.on_click(add_clicked)
    plot_button.on_click(plot_clicked)

    title = HTML(
        value=f'<div style="overflow-wrap:anywhere; word-break:break-word"><b>{html_module.escape(str(siteid))}</b> ({html_module.escape(str(org))})</div>',
        layout=wg.Layout(width='100%', max_width='100%', overflow='hidden'),
    )
    buttons = HBox(
        [add_button, plot_button],
        layout=wg.Layout(width='100%', justify_content='space-between', overflow='hidden'),
    )
    return VBox(
        [title, buttons, metadata],
        layout=wg.Layout(width='100%', max_width='100%', overflow='hidden', padding='2px'),
    )


def plot_site_locations(map_obj: object, org: str, coordslist: list, siteidlist: list, tooltiplist: list, color: str = "blue", dotrad: int = 3, popup_owner: object = None) -> None:
    """Plots station markers with station-aware popups on an ipyleaflet map.

    Args:
        map_obj: ipyleaflet map or layer group
        org: data source name
        coordslist: list of latitude and longitude pairs
        siteidlist: list of station codes
        tooltiplist: list of popup labels
        color: marker color
        dotrad: marker radius in pixels
    """
    # Attach widgets directly to markers, like the demo. This lets Leaflet size
    # the popup from the real controls instead of constraining a widget inside a Popup layer.
    popup_target = resolve_popup_owner(map_obj, popup_owner)
    for coords, siteid, tooltip in zip(coordslist, siteidlist, tooltiplist):
        marker = CircleMarker(
            location=coords,
            pane=STATION_PANE_NAME,
            radius=dotrad,
            color=color,
            weight=0,
            fill_color=color,
            fill_opacity=1,
            opacity=1,
        )

        # Build one Popup layer lazily. Hover primes the cache; click explicitly opens it.
        def load_popup(_event=None, marker=marker, siteid=siteid, org=org, tooltip=tooltip, **_kwargs):
            """Prime a cached popup on hover and open it on the first click."""
            popup_layer = getattr(marker, "_gnss_popup_layer", None)
            if popup_layer is None:
                popup_layer = Popup(
                    location=list(marker.location),
                    child=station_popup(siteid, org, tooltip),
                    min_width=280,
                    max_width=460,
                    max_height=360,
                )
                # Keep the child on the marker for ipyleaflet's normal binding,
                # while the Popup layer provides a reliable explicit open call.
                marker.popup = popup_layer.child
                marker._gnss_popup_layer = popup_layer
            event_type = _kwargs.get("type") if isinstance(_kwargs, dict) else None
            if isinstance(_event, dict):
                event_type = _event.get("type", event_type)
            if event_type == "click":
                if popup_layer not in getattr(popup_target, "layers", ()):
                    add_map_layer(popup_target, popup_layer)
                popup_layer.open_popup(marker.location)

        marker.on_mouseover(load_popup)
        marker.on_click(load_popup)
        add_map_layer(map_obj, marker)

def nearby_sites(org: str, site: object, radius_in_km: float) -> tuple:
    """Finds stations within a given distance of a site or coordinate pair.

    Args:
        org: data source name
        site: station code or coordinate pair
        radius_in_km: search radius in kilometers

    Returns:
        A tuple containing nearby station records and the center coordinate pair.
    """
    if isinstance(site, str):
        site = site.strip()
        if site[0] in "([": # if it's a coordinate pair instead of a site id
            coords = ast.literal_eval(site)
            if not isinstance(coords, (list, tuple)) or len(coords) != 2:
                raise ValueError("Coordinates must contain latitude and longitude")
            site = [float(value) for value in coords]
        else:
            site = site.upper()

    cur_coords = site

    if isinstance(site,str):
        cur_coords = data_of[org][site]["location"]

    assert not isinstance(cur_coords[0],str) # by now cur_coords is a coordinate pair regardless of input

    id_coord_dist_list = []

    for siteid in data_of[org]:
        try:
            site_coords = data_of[org][siteid]["location"]
            distance = d_km(site_coords, cur_coords)
            if distance < radius_in_km : #  and site != siteid:  (siteid site as well for tooltip)
                id_coord_dist_list.append((siteid,*site_coords,distance))
        except: pass

    return id_coord_dist_list, cur_coords


def basic_circle(map_obj: object, center: list, radius_in_km: float) -> None:
    """Draws a radius circle around a center point on the map.

    Args:
        map_obj: ipyleaflet map or layer group
        center: center latitude and longitude pair
        radius_in_km: circle radius in kilometers
    """
    circle = Circle(
        location=center,
        radius=radius_in_km * 1000, # in metres
        color="black",
        weight=1,
        fill_opacity=0.0,
        opacity=1,
        fill_color="green",
        fill=True,
    )

    add_map_layer(map_obj, circle)


def vec_add(c1,c2):
    return core_vec_add(c1, c2)

def vec_sub(c1,c2):
    return core_vec_sub(c1, c2)

def velocity_endpoint(center: list, direction: list, scale: float) -> list:
    """Calculates the map endpoint for a North/East velocity vector.

    Args:
        center: starting latitude and longitude pair.
        direction: velocity vector as North, East, and Up components.
        scale: map scale in kilometers per millimeter per year.

    Returns:
        A latitude and longitude pair for the vector endpoint.
    """
    return core_velocity_endpoint(center, direction, scale)


def draw_vector(map_obj: object, center: list, direction: list, scale: float, color: str = "black", label: str = "None", *, org: str = "", site: str = "", sigma: list = (), siglim: list = (), up_color_factor: float = 0.0, relative: bool = False) -> None:
    """Draw one fresh screen-space SVG vector layer.

    Absolute/source vectors reuse an immutable core render spec; every call
    still instantiates new DivIcon and Marker objects for its map owner.
    Relative vectors bypass the cache because their reference can change.
    Each fresh DivIcon receives explicit transparent classes
    ``gnss-vector-div-icon`` and ``gnss-vector-label-div-icon``.
    """
    if relative:
        # Relative vectors bypass the absolute/source LRU cache.
        render_spec = relative_vector_render_spec(
            tuple(float(value) for value in center[:2]), tuple(float(value) for value in direction[:3]),
            tuple(float(value) for value in (sigma or (0, 0, 0))[:3]), tuple(float(value) for value in (siglim or (0, 0, 0))[:3]),
            float(up_color_factor), float(scale), color, label,
        )
    else:
        sigma_values = tuple(float(value) for value in (sigma or (0, 0, 0))[:3])
        siglim_values = tuple(float(value) for value in (siglim or (0, 0, 0))[:3])
        render_spec = absolute_vector_render_spec(
            str(org), str(site), tuple(float(value) for value in center[:2]),
            tuple(float(value) for value in direction[:3]), sigma_values, siglim_values,
            float(up_color_factor), float(scale), str(color), label,
        )
    add_map_layer(map_obj, Marker(
        location=list(render_spec.location),
        pane=VECTOR_PANE_NAME,
        keyboard=False,
        icon=_vector_div_icon(render_spec.svg, render_spec.icon_size, render_spec.icon_anchor, class_name='gnss-vector-div-icon'),
    ))
    if render_spec.label_html is not None:
        add_map_layer(map_obj, Marker(
            location=list(render_spec.location),
            pane=VECTOR_PANE_NAME,
            keyboard=False,
            icon=_vector_div_icon(render_spec.label_html, render_spec.label_icon_size, render_spec.label_icon_anchor, class_name='gnss-vector-label-div-icon'),
        ))


### Timeseries Functions

In [244]:
def remove_brac(string: str) -> tuple:
    """Splits a display label into station code and data source.

    Args:
        string: label formatted like "P123 (UNR)"

    Returns:
        A tuple containing the station code and data source.
    """
    return core_remove_brac(string)


def plot_ts_graph(siteid: str, org: str, Update: bool, detrend: bool, ax0: object, ax1: object, ax2: object, yearrange: list, breaks: bool, errorbars: list, errorbar_outline: list, outlier: float, shift: float, color: str) -> object:
    """Adds one station time series to the three-component plot.

    Args:
        siteid: 4-character station code
        org: data source name
        Update: whether to download a fresh copy of the time series
        detrend: whether to remove the fitted linear trend
        ax0: matplotlib axis for North displacement
        ax1: matplotlib axis for East displacement
        ax2: matplotlib axis for Up displacement
        yearrange: start and end datetime values for the plot
        breaks: whether to remove known break offsets
        errorbars: error bar settings
        errorbar_outline: filled error outline settings
        outlier: outlier removal multiplier, or 0 to skip removal
        shift: vertical display shift to apply to this station
        color: plot color

    Returns:
        A DataFrame of fitted velocity statistics when detrending is enabled; otherwise an empty list.
    """

    try:
        times, tseries = Read[org](siteid, Update)
    except:
        print('Error getting',siteid,'from',org)
        times = []; tseries = []
        return

    td, nd, ns, ed, es, ud, us = tseries

    keep = np.array([yearrange[0] <= time <= yearrange[1] for time in times])
    times, td = np.asarray(times)[keep], td[keep]
    nd, ns = nd[keep], ns[keep]
    ed, es = ed[keep], es[keep]
    ud, us = ud[keep], us[keep]
    if len(times) == 0:
        print("No data for", siteid, "in the selected date range")
        return []

    if breaks:
        if "breaks" in data_of[org][siteid]:
            breaktimes_dict = {}
            for dt in data_of[org][siteid]["breaks"]:
                try:
                    yr_mm_day = [int(date) for date in dt[1:-1].split(",")][0:5]
                    breaktime = datetime(*yr_mm_day)
                    breaktimes_dict[breaktime] = [data_of[org][siteid]["breaks"][dt]["offsets"], int(np.where(times >= breaktime)[0][0])]
                except:
                    pass
            # [45.01, 0.67, 39.72, 0.59, -27.67, 1.83], [dN (mm), sN (mm), dE (mm), sE (mm), dU (mm), sU (mm)]
            interval_counter = 0
            breaktimes_list = list(breaktimes_dict)
            ndat = len(ud)
            while interval_counter in range(len(breaktimes_dict)):
                dict_key = breaktimes_list[interval_counter]
                interval_start = breaktimes_dict[dict_key][1]
                # Using -1 as last point skipped last point
                offsets = breaktimes_dict[dict_key][0]
                nd[interval_start:ndat] = nd[interval_start:ndat] - offsets[0]
                ed[interval_start:ndat] = ed[interval_start:ndat] - offsets[2]
                ud[interval_start:ndat] = ud[interval_start:ndat] - offsets[4]
                ns[interval_start:ndat] = np.hypot(ns[interval_start:ndat], offsets[1])
                es[interval_start:ndat] = np.hypot(es[interval_start:ndat], offsets[3])
                us[interval_start:ndat] = np.hypot(us[interval_start:ndat], offsets[5])
                interval_counter +=1
        else:
            print("No breaks information!")

    dfa = []
    if detrend:
        nd, VelN, StatN = detrended(td, nd, ns)
        ed, VelE, StatE = detrended(td, ed, es)
        ud, VelU, StatU = detrended(td, ud, us)

        if outlier: # while length of the array before and after are different, keep iterating the code
            initial_length = len(nd)
            length1 = 1
            length2 = 2
            iteration_cnt = 0
            while length1 != length2:
                length1 = len(nd)
                nd, ed, ud, ns, es, us, times, td = remove_outliers_function(nd, ed, ud, ns, es, us, times, td, outlier)
                nd, dVelN, StatN = detrended(td, nd, ns)
                VelN[0] = VelN[0]+dVelN[0]   # Fit is to residuals, so update total

                ed, dVelE, StatE = detrended(td, ed, es)
                VelE[0] = VelE[0]+dVelE[0]   # Fit is to residuals, so update total
                ud, dVelU, StatU = detrended(td, ud, us)
                VelU[0] = VelU[0]+dVelU[0]   # Fit is to residuals, so update total

                length2 = len(nd)
                iteration_cnt +=1
            print(f"Final {len(nd)} data; Number of outliers removed: {initial_length - len(nd)} with {iteration_cnt} iterations: ")

        VelNEU = VelN+VelE+VelU ; StatNEU = StatN[0:2]+StatE[0:2]+StatU
        labl = str(siteid)+'-'+str(org)
        dfa = pd.DataFrame([VelNEU+StatNEU], \
            columns=['Vn','σVn','Ve','σVe','Vu','σVu','WRMS N','χn','WRMS E','χe','WRMS U','χu','Num'],index=[labl])


    nd = nd+shift ;  ed = ed+shift ; ud = ud+shift*3

    if errorbars[0]:
        ax0.errorbar(times,nd,yerr=[ns,ns],errorevery=errorbars[1], capsize=2, color = color, linewidth=1) #, ecolor='black')
        ax1.errorbar(times,ed,yerr=[es,es],errorevery=errorbars[1], capsize=2, color = color, linewidth=1) #, ecolor='black')
        ax2.errorbar(times,ud,yerr=[us,us],errorevery=errorbars[1], capsize=2, color = color, linewidth=1) #, ecolor='black')

    if errorbar_outline[0]:
        for ax, d, s in [(ax0, nd, ns), (ax1, ed, es), (ax2, ud, us)]:
            ax.fill(list(times) + list(reversed(times)),
                list(d+s) + list(np.flip(d-s)),
                alpha=errorbar_outline[1], linewidth=1, color = color, label="_"+siteid)

    for ax, d in [(ax0, nd), (ax1, ed), (ax2, ud)]:
        ax.plot(times, d, linewidth=0.5, label=siteid + " (" + org + ")", color = color)
        if "breaks" in data_of[org][siteid]:
            for dt in data_of[org][siteid]["breaks"]:
                yr_mm_day = [int(date) for date in dt[1:-1].split(",")][0:3]
                breaktime = datetime(*yr_mm_day)
                try:
                    np.where(times >= breaktime)[0][0]
                    ax.axvline(x=breaktime, color='r', ls='--')
                except:
                    pass

    return dfa

def plot_ts_graph_list(idlist: list, Update: bool, yearrange: list, breaks: bool, detrend: bool, errorbars: list, errorbar_outline: list, outlier: float, resolution: str = "Low Res", shift: object = 0) -> None:

    """Plots North, East, and Up time series for a list of selected stations.

    Args:
        idlist: selected station labels formatted like "P123 (UNR)"
        Update: whether to download fresh time-series files
        yearrange: start and end datetime values for the plot
        breaks: whether to remove known break offsets
        detrend: whether to remove fitted linear trends
        errorbars: error bar settings
        errorbar_outline: filled error outline settings
        outlier: outlier removal multiplier, or 0 to skip removal
        resolution: figure resolution setting
        shift: dictionary of display shifts by station label
    """

    global tsfig

    tsfig = plt.figure(figsize=(15, 10), dpi = {"HD": 600, "Regular": 300, "Low Res": 100}[resolution])
    gs = gridspec.GridSpec(3, 1, height_ratios=[1, 1, 1])

    ax0 = plt.subplot(gs[0])
    ax0.set_ylabel("ΔNorth (mm)")
    ax0.yaxis.set_tick_params(labelrotation=90)
    plt.setp(ax0.get_xticklabels(), visible=False)

    ax1 = plt.subplot(gs[1], sharex = ax0)
    ax1.set_ylabel("ΔEast (mm)")
    ax1.yaxis.set_tick_params(labelrotation=90)
    plt.setp(ax1.get_xticklabels(), visible=False)

    ax2 = plt.subplot(gs[2], sharex = ax0)
    ax2.set_ylabel("ΔUp (mm)")
    ax2.yaxis.set_tick_params(labelrotation=90)
    ax2.set_xlabel("Time")

    colors = ["b", 'g', 'r', 'c', 'm', 'y', 'k', 'sienna']
    color_count = 0
    for idorg in idlist:
        dfl = plot_ts_graph(*remove_brac(idorg), Update, detrend, ax0, ax1, ax2, yearrange, breaks, errorbars, errorbar_outline, outlier, shift[idorg] if idorg in shift else 0, colors[color_count])
        if color_count < 8:
            color_count += 1
        else:
            color_count = 0

        if detrend:
            if color_count == 1:
                dfall = dfl
            else:
                dfall = pd.concat([dfall, dfl])

    if detrend:
        display(dfall.style.format({'Vn': '{:.2f}','σVn': '{:.3f}',
                                'Ve': '{:.2f}','σVe': '{:.3f}',
                                'Vu': '{:.2f}','σVu': '{:.3f}',
                                'WRMS N': '{:.2f}','χn': '{:.2f}',
                                'WRMS E': '{:.2f}','χe': '{:.2f}',
                                'WRMS U': '{:.2f}','χu': '{:.2f}','Num':'{:d}'}))


    leg = ax0.legend()

    plt.subplots_adjust(hspace=.0)

    if mpl.get_backend() == 'nbagg':
        from IPython.display import display as ipdisplay
        ipdisplay(tsfig)
    else:
        plt.show()


### Widgets

#### Fetcher Widgets

In [ ]:
# Panel system: palette, stylesheet, and container helpers for the four
# non-map windows. Reloaded like gnss_core so edits land without a restart.
import gnss_ui
importlib.reload(gnss_ui)
ui = gnss_ui

# Buttons
update_UNR_butt     = wg.Button(description="Latest UNR", icon = "download")
update_JPL_butt     = wg.Button(description="Latest JPL", icon = "download")
update_NGF_butt     = wg.Button(description="Latest NGF", icon = "download")
breaks_select       = wg.Dropdown(options=orglist, value='UNR', disabled=False)
add_breaks_site_button = wg.Button(description="Add Breaks", icon = "plus-square")
remove_breaks_site_button = wg.Button(description="Remove Breaks", icon = "minus-square")
clear_fetch_log_butt = wg.Button(description="Clear Log", icon = "times")

# Data location
data_root_label      = wg.HTML(layout=wg.Layout(width='100%', min_width='0', margin='0'))
data_root_input      = wg.Text(placeholder='/path/to/data', continuous_update=False)
data_root_browser    = wg.Select(options=[], rows=6,
                                 layout=wg.Layout(width='100%', min_width='0', margin='0'))
data_root_use_butt   = wg.Button(description="Use This Folder", icon="check")
data_root_reset_butt = wg.Button(description="Reset to Default", icon="undo")
local_file_input     = wg.Text(placeholder='/path/to/SITE.cwu.igs14.csv', continuous_update=False)
add_local_file_butt  = wg.Button(description="Add Local File", icon="file")

# Outputs
fetcher_output      = wg.Output()
breaks_update_output = wg.Output()
data_location_output = wg.Output()


#### Availability Widgets

In [ ]:
# Inputs
site_searchbar      = wg.Text(value='', placeholder='e.g. P040', disabled=False, continuous_update=False)
org_avail_select    = wg.Dropdown(options=orglist, value='NGF', disabled=False)
site_search_submit  = wg.Button(description="Search", icon = "search")
clear_log_butt      = wg.Button(description="Clear Log", icon = "times")
# Readouts
availability_detail = wg.HTML(layout=wg.Layout(width='100%', min_width='0', margin='0'))
# Outputs
availability_output = wg.Output()

#### Map Widgets

In [ ]:
## TODO: vector controls changed to sliders for easier scale adjustment
# Inputs
map = ["Not initialized yet!"] # allow direct editing of the map via entry mutation
map_dynamic_layers = [None] # stations, circles, and vectors that can be replaced without remounting the map
map_reloading = [False] # prevent a second reload while the current one is still building layers
map_loading = HTML(
    value='<div style="background:rgba(255,255,255,0.92); border:1px solid #888; padding:8px 12px; font-weight:600">Updating map...</div>',
    layout=wg.Layout(display='none')
)
layout              = lambda w: wg.Layout(width=w, height='40px')
new_map_butt        = wg.Button(description="Clear / New Map", icon = "map")
reload_map_butt     = wg.Button(description="Reload Map", icon = "refresh")
close_map_butt      = wg.Button(description="Close Map", icon = "times")
station_field_layout = wg.Layout(flex='1 1 0')
site_id         = wg.Text(value=None,placeholder='e.g. P049',disabled=False, layout=station_field_layout)
org_map_select      = wg.Dropdown(options=orglist+['other'], value='NGF', disabled=False, layout=station_field_layout)
site_radius         =site_radius = wg.Text(
    value='',
    placeholder='0',
    disabled=False,
    layout=station_field_layout
)
vel_siglim_form     = wg.Text(
    value=None, description='Deviation Limit (σ):', placeholder='(1,1,2)',
    style={'description_width': 'initial'}, layout=wg.Layout(width='88%')
)
plot_vec_check      = wg.ToggleButton(description="Plot Velocities", value=False, icon='long-arrow-right', layout = layout("200px"))
velocity_mode_select = wg.Dropdown(
    options=[('Absolute', 'source'), ('Relative', 'relative')],
    value='source', disabled=False, layout=wg.Layout(width='68%')
)
arrow_length_input    = wg.FloatSlider(description = "Guide mm/yr", min = 1, max = 50, step = 1, value = 10,
                                      readout_format = '.0f', layout = wg.Layout(width='82%'),
                                      tooltip="Magnitude represented by the screen-fixed velocity reference bracket")
velocity_scale_input   = wg.FloatSlider(description = "Vel Scale", min = 1, max = 30, step = 1, value = 10,
                                       readout_format = '.0f', layout=wg.Layout(width='82%'),
                                       tooltip='Controls plotted-vector and guide length conversion; the geographic-equivalent readout varies with live zoom and latitude')
colorscalefactor_input = wg.FloatSlider(description = "±U Rate", min = 1, max = 20, step = 1, value = 5,
                                       readout_format = '.0f', layout=wg.Layout(width='82%'),
                                       tooltip='±<Range for Vertical Rate> (color saturates after this value) red (negative) to blue (positive)')
thin_velocities_input  = wg.IntSlider(description = "Decimate", min = 1, max = 50, step = 1, value = 1,
                                     layout=wg.Layout(width='82%'))

# map radius lists
map_radius_list     = wg.SelectMultiple(options=[], value=[], description = "Site/Radius:", layout=wg.Layout(display='none'))
table_content_width = '99%'
bulk_selection_control_height = '2rem'
bulk_selection_input = wg.Text(
    placeholder='e.g. 1, 2, [4, 8], 12',
    layout=wg.Layout(width='78%', height=bulk_selection_control_height, min_height=bulk_selection_control_height)
)
select_bulk_sites_butt = wg.Button(description='Select', layout=wg.Layout(width='20%', height=bulk_selection_control_height, min_height=bulk_selection_control_height))
select_all_sites_butt = wg.Button(description='Select All', layout=wg.Layout(width='100%', height=bulk_selection_control_height, min_height=bulk_selection_control_height))
map_site_scroll_output = wg.Output(layout=wg.Layout(display='none'))
site_table          = VBox(
    [],
    layout=wg.Layout(width=table_content_width, min_width='0', box_sizing='border-box', margin='0.5% 0 0 0', flex='0 0 auto')
)
neighbor_table = VBox(
    [],
    layout=wg.Layout(
        width=table_content_width, min_width='0', margin='0 0 0.5% 0', flex='0 0 auto',
        overflow_x='hidden', overflow_y='hidden',
    )
)
table_spacer        = wg.Box([], layout=wg.Layout(
    height='20px', min_height='20px', max_height='20px', flex='0 0 20px'
))
map_site_checks     = {}
add_sites_map_button  = wg.Button(description = "Add Site", icon='plus-circle')
table_action_layout = wg.Layout(width='210px', height='40px')
remove_sites_map_button = wg.Button(description = "Remove Site(s)", icon='minus-circle', layout=table_action_layout)
clear_map_list_butt     = wg.Button(description="Clear List", icon = "times", layout=table_action_layout)

site_circle_submit  = wg.Button(description="Plot Site(s)", icon = "map-marker", layout=table_action_layout)

# Style
# Remove Site(s) and Clear List are different severities, not one destructive
# group: Remove acts only on what's checked (selective, easy to redo by
# re-adding); Clear List wipes the whole table at once. Keeping them visually
# distinct is what lets you tell them apart at a glance before clicking.
for neutral_map_butt in [new_map_butt, reload_map_butt, close_map_butt,
                         select_all_sites_butt, select_bulk_sites_butt,
                         remove_sites_map_button]:
    neutral_map_butt.style.button_color = '#ECEEF1'
    neutral_map_butt.style.text_color = '#1A1D21'
for primary_map_butt in [site_circle_submit, add_sites_map_button]:
    primary_map_butt.style.button_color = '#1966FF'
    primary_map_butt.style.text_color = '#FFFFFF'
    primary_map_butt.style.font_weight = '600'
for destructive_map_butt in [clear_map_list_butt]:
    destructive_map_butt.style.button_color = '#FDECEA'
    destructive_map_butt.style.text_color = '#B42318'
    destructive_map_butt.style.font_weight = '600'
    destructive_map_butt.add_class('gnss-btn-danger')
# ToggleButton.style carries no button_color trait, so the velocity toggle's
# active state is accented through the panel stylesheet instead.
plot_vec_check.add_class('gnss-accent-toggle')

# Basemap options
BaseMap_butt = wg.Dropdown(
    options=['OpenTopo', 'OpenStreetMap', 'ArcGIS Image'],
    value='OpenTopo',
    disabled=False,
    layout=wg.Layout(flex='1 1 0')
)
BaseMap_Set = wg.Button(description = "Set Base Map")


# Outputs
map_output          = wg.Output()
nearest_site_output = wg.Output()
graph_output        = wg.Output()

#### Timeseries Widgets

In [ ]:
# Inputs
ts_site_form        = wg.Text(value='', placeholder='e.g. P040', disabled=False)
org_ts_select       = wg.Dropdown(options=orglist+['other'], value='NGF', disabled=False)
append_butt         = wg.Button(description="Add to List", icon = "plus-square")
plot_ts_butt        = wg.Button(description="Plot", icon = "line-chart")
plot_ts_res         = wg.Dropdown(options=['HD', 'Regular', 'Low Res'], value='Low Res', disabled=False)
close_ts_butt       = wg.Button(description="Close Graph", icon = "times")
clear_list_butt     = wg.Button(description="Clear List", icon = "times")
ts_sites            = wg.SelectMultiple(options=[], value=[], rows=9,
                                        layout=wg.Layout(width='100%', min_width='0', margin='0'))
ts_filter_form      = wg.Text(value='', placeholder='e.g. P04 or UNR', disabled=False)

# TS customizations

error_bar_check     = wg.Checkbox(description="Error Bars", value=False, indent=False)
error_bar_outline_check = wg.Checkbox(description = "Error Bar Outlines", value = False, indent=False)

detrend_check       = wg.Checkbox(description="Detrend", value=False, indent=False)
live_update_check   = wg.Checkbox(description="Live Update", value=False, indent=False)
start_year_form     = wg.Text(value='', placeholder='YYYY-MM-DD', disabled=False)
end_year_form       = wg.Text(value='', placeholder='YYYY-MM-DD', disabled=False)
siglim_form         = wg.Text(value='', placeholder='(10,10,30)', disabled=False)

shift_value        = wg.BoundedFloatText(value=0.0, min=-500.0, max=500.0, step=0.5, disabled=False)
shift_slider       = wg.FloatSlider(value=0.0, min=-500.0, max=500.0, step=0.5, readout=False)
remove_site_button  = wg.Button(description="Remove Site", icon = "minus-square")
update_customization = wg.Button(description="Update shift", icon = "arrow-up")
shift_output       = wg.Output()

thin_error_bars    = wg.BoundedIntText(value=1, min=1, max=100, step=1, tooltip='Show every Nth error bar')
thin_error_bars_slider = wg.IntSlider(value=1, min=1, max=100, step=1, readout=False)
error_bar_opacity  = wg.BoundedFloatText(value=0.1, min=0.0, max=1.0, step=0.05)
error_bar_opacity_slider = wg.FloatSlider(value=0.1, min=0.0, max=1.0, step=0.05, readout=False)
add_breaks_ts_button = wg.Button(description = "Copy Break Data")
remove_breaks_ts_button = wg.Button(description="Clear Break Data")

remove_outliers = wg.Dropdown(options=[None, 1, 2, 3, 4, 5, 6], value = None, disabled = False)
show_breaks_data_button = wg.Button(description = "Show Breaks Data")
remove_breaks_checkbox = wg.Checkbox(description = "Remove Breaks", value = False, indent=False)
breaks_data_output = wg.Output()

# ID points option (only in iterative mode)
id_button =  wg.Button(description = "ID points",tooltip='right-click/return to end\nmiddle/delete to remove')

# Graphics Backends
backend_butt = wg.RadioButtons(
    options=['Inline', 'ipympl', 'Interactive'],
    value='Inline',
    orientation='horizontal',
    disabled=False
)
backend_activate = wg.Button(description = "Backend Activate")

# Live readouts (filled by refresh_ts_state in the windows cell)
ts_preview         = wg.HTML(layout=wg.Layout(width='100%', min_width='0', margin='0'))
ts_ribbon          = wg.HTML(layout=wg.Layout(width='100%', min_width='0', margin='0 0 0.5rem 0'))
ts_results_ribbon  = wg.HTML(layout=wg.Layout(width='100%', min_width='0', margin='0 0 0.5rem 0'))

# Outputs
timeseries_output = wg.Output()

### Widget Functions

#### Database Fetcher

In [ ]:
## TODO: NGF update auth branch now opens EarthScope portal when needed
def UpdateUNR(_):
    with fetcher_output:
        try:
            print("Downloading Data from UNR...")

            # Updated to file location URL
            plates_site = "https://geodesy.unr.edu/gps_timeseries/Plates/sta_frames.txt"
            coords_site = "https://geodesy.unr.edu/NGLStationPages/llh.out"
            # Updated to IGS14 TAH 260330.
            vels_site   = "https://geodesy.unr.edu/velocities/midas.IGS14.txt"

            ## TODO: Explore replacing with decode to avoid weird formatting issues if website changes
            plates_list = str(urllib.request.urlopen(plates_site).read())[2:].split(" \\n")[:-1]
            coords_list = str(urllib.request.urlopen(coords_site).read())[2:].split(" \\n")[:-1]

            for_json = {}

            for siteplate in plates_list:
                siteplates = [el for el in siteplate.split(" ") if el]
                for_json.setdefault(siteplates[0], {'location': None, 'height': [], 'regions': []})
                for_json[siteplates[0]]['regions'] = siteplates[1:]

            for sitecoord in coords_list:
                siteid, lon, lat, height = (el for el in sitecoord.split(" ") if el)
                for_json.setdefault(siteid, {'location': None, 'height': [], 'regions': []})
                for_json[siteid]['location'] = [float(lon), float(lat)]
                for_json[siteid]['height'] = float(height)

            ## TODO: Add reading UNR velocities
            df = pd.read_csv(vels_site,delimiter=r"\s+",usecols=[0,8,9,10,11,12,13])
            nvel = len(df) ; print("Processing",nvel,"lines")
            for i in range(nvel):
                siteid = df.iloc[i,0]
                for_json.setdefault(siteid, {})
                for_json[siteid]["velocity"] = list(df.iloc[i,[2,1,3]]*1000) # switch to mm and from ENU to NEU
                for_json[siteid]["velsig"] = list(df.iloc[i,[5,4,6]]*1000)


            global data_of

            data_of["UNR"] = for_json

            with open(DATA_DIR / 'UNR' / 'UNR_data.json', 'w') as f:
                json.dump(for_json, f)

            print("Latest UNR Data Downloaded")
        except Exception as update_error:
            print('UNR update failed: %s: %s' % (type(update_error).__name__, update_error))

update_UNR_butt.on_click(UpdateUNR)


def UpdateJPL(_):
    with fetcher_output:
        try:
            print("Downloading Data from JPL...")

            coords_site = "https://sideshow.jpl.nasa.gov/post/tables/table2.html"
            x = str(urllib.request.urlopen(coords_site).read()).split("\\n")
            sitecoordlist = [[xxx for xxx in xx.split(" ") if xxx]
                             for xx in x if "POS" in xx]
            sitevellist = [[xxx for xxx in xx.split(" ") if xxx]
                             for xx in x if "VEL" in xx]
            # [['AB01', 'POS', '52.2095', '-174.2048', '25492.217', '0.034', '0.024', '0.098'], ...]
            for_json = {}
            for element in sitecoordlist:
                for_json[element[0]] = {'location': [float(element[2]),float(element[3])],'height': float(element[4])/1000.0}
            for element in sitevellist:
                for_json[element[0]]['velocity'] = [float(element[2]),float(element[3]), float(element[4])]
                for_json[element[0]]['velsig'] = [float(element[5]),float(element[6]), float(element[7])]

            global data_of
            data_of["JPL"] = for_json

            with open(DATA_DIR / 'JPL' / 'JPL_data.json', 'w') as f:
                json.dump(for_json, f)

            print("Latest JPL Data Downloaded")
        except Exception as update_error:
            print('JPL update failed: %s: %s' % (type(update_error).__name__, update_error))

update_JPL_butt.on_click(UpdateJPL)


def UpdateNGF(_):
    with fetcher_output:
        try:
            print("Downloading Data from NGF...")

            coords_site = "https://www.unavco.org/instrumentation/networks/status/data/geoJSON/network-monitoring"

            x = json.loads(urllib.request.urlopen(coords_site).read())
            print('Back from json.load ')
            # [['AB01', 'POS', '52.2095', '-174.2048', '25492.217', '0.034', '0.024', '0.098'], ...]
            for_json = {}
            for element in x['features']:
                for_json[element["id"]] = {
                    'location': [element['geometry']['coordinates'][1],element['geometry']['coordinates'][0]],
                    'height': float(element['properties']['elev']),
                    'region': element['properties']['region'],
                    'stntype': element['properties']['stntype']}

            # Read Breaks #
            if ( esver[0] == '0' ):
                get_es_file("https://gage-data.earthscope.org/archive/gnss/products/offset/cwu.kalts_nam14.off")
            else:
                try:
                    from earthscope_sdk import EarthScopeClient
                    from earthscope_sdk.auth.auth_flow import NoAccessTokenError
                    es_client = EarthScopeClient()
                    # The SDK returns a cached access token even once it has expired,
                    # and the archive rejects that with HTTP 401. Renew it first; the
                    # stored grant includes offline_access, so this needs no login.
                    try:
                        es_client.ctx.auth_flow.refresh_if_necessary()
                    except Exception as refresh_error:
                        print("Could not refresh the EarthScope token: %s" % refresh_error)
                        print("Run `es login` in a terminal, then press Latest NGF again.")
                        return
                    token = es_client.ctx.auth_flow.access_token
                    headers = {"Authorization": f"Bearer {token}"}
                except NoAccessTokenError:
                    print("EarthScope authentication required. Opening the EarthScope data portal...")
                    webbrowser.open("https://data.earthscope.org")
                    print("If the browser login does not create a token, run `es login` in your terminal, then re-run this cell.")
                    return
                fname = "cwu.kalts_nam14.off"
                outpath = DATA_DIR / "NGF" / "cwu.kalts_nam14.off"
                url = "https://gage-data.earthscope.org/archive/gnss/products/offset/cwu.kalts_nam14.off"
                print('Calling fetch_csv with ',url,fname)
                url, ok, err = fetch_csv(url, outpath, headers=headers)
                if ok:
                    print(f"✓ {fname}")
                else:
                    print(f"✗ {fname}  --  {err}")
                    if "401" in str(err):
                        print("EarthScope rejected the token. Run `es login` in a "
                              "terminal, then press Latest NGF again.")
                    print("NGF update stopped: the offsets file was not downloaded, so "
                          "there is nothing to read.")
                    return

            with open(DATA_DIR / "NGF" / "cwu.kalts_nam14.off", "r") as f:
                data = list(f)

            i = 0
            while 'End Field Description' not in data[i]: i += 1

            data = data[i+1:]

            for i in range(1, len(data)):
                info = data[i].split()
                siteid = info[0]
                for_json.setdefault(siteid, {})
                dt = [int(pt) for pt in info[1:6]]
                moment = str(tuple(dt[0:5]))
                for_json[siteid].setdefault("breaks", {})
                for_json[siteid]["breaks"][moment] = {"offsets": [float(pt) for pt in info[6:12]],
                                                      "type": info[12],
                                                      "description": " ".join(info[13:]),}
            ###############

            # Read Velocities MOD TAH: Replace final with snaps file.
            for velf in ('cwu.snaps_igs14.vel','cwu.fanet_igs14.vel') :
                if ( esver[0] == '0' ):
                    get_es_file("https://gage-data.earthscope.org/archive/gnss/products/velocity/"+velf)
                # https://gage-data.earthscope.org/archive/gnss/products/velocity/cwu.fanet_igs14.vel
                else :
                    fname = velf
                    outpath = DATA_DIR / "NGF" / velf
                    url = "https://gage-data.earthscope.org/archive/gnss/products/velocity/"+velf
                    print('Calling fetch_csv with ',url,fname)
                    url, ok, err = fetch_csv(url, outpath, headers=headers)
                    if ok:
                        print(f"✓ {fname}")
                    else:
                        print(f"✗ {fname}  --  {err}")
                        if "401" in str(err):
                            print("EarthScope rejected the token. Run `es login` in a "
                                  "terminal, then press Latest NGF again.")
                        print("NGF update stopped: %s was not downloaded." % fname)
                        return


                with open(DATA_DIR / "NGF" / velf, "r") as f:
                    data = list(f)

                i = 0
                while 'End Field Description' not in data[i]: i += 1

                data = data[i+1:]

                for i in range(1, len(data)):
                    info = data[i].split()
                    siteid = info[0]
                    for_json.setdefault(siteid, {})
                    for_json[siteid]["velocity"] = [float(pt)*1000 for pt in info[19:22]] # switch to mm
                    for_json[siteid]["velsig"] = [float(pt)*1000 for pt in info[22:25]]
                    # Get positions from velocity file 8,9,10 lat, long, height
                    if( float(info[8])>180 ):
                        for_json[siteid]["location"] = [float(info[7]),float(info[8])-360]
                    else:
                        for_json[siteid]["location"] = [float(info[7]),float(info[8])]

                    for_json[siteid]["height"] = float(info[9])


            ###############

            global data_of
            data_of["NGF"] = for_json

            with open(DATA_DIR / 'NGF' / 'NGF_data.json', 'w') as f:
                json.dump(for_json, f)

            print("Latest NGF Data Downloaded")
        except Exception as update_error:
            print('NGF update failed: %s: %s' % (type(update_error).__name__, update_error))

update_NGF_butt.on_click(UpdateNGF)


#### Availability

In [ ]:
def site_search(_button):
    searched = site_searchbar.value.strip().upper()
    with availability_output:
        if not data_of.get(org_avail_select.value):
            display(wg.HTML(
                '<em style="color:#B42318">' + org_avail_select.value + ' catalog is '
                'not loaded. Press "Latest ' + org_avail_select.value + '" in the '
                'Database Fetcher window first.</em>'))
            return
        try:
            print(searched + "(" + org_avail_select.value + ")", ": ")
            org = org_avail_select.value
            long_tuple = data_of[org][searched]
            long_string = str(long_tuple)
            wrapped_string = textwrap.wrap(long_string, width=150)
            for line in wrapped_string:
                print(line)
        except:
            display(wg.HTML('''<em style="color:red">Not Found!</em>'''))

site_search_submit.on_click(site_search)


def clear_avail_log(_button):
    availability_output.clear_output()
clear_log_butt.on_click(clear_avail_log)

def add_breaks_data_from_json(_button):
    org = breaks_select.value
    clear_absolute_vector_render_cache()
    if not data_of.get(org) or not data_of.get("NGF"):
        with breaks_update_output:
            display(wg.HTML(
                '<em style="color:#B42318">Needs both the NGF catalog (the source of '
                'break data) and the ' + org + ' catalog loaded. Fetch them in the '
                'Database Fetcher window first.</em>'))
        return
    for site in data_of[org]:
        if site in data_of["NGF"]  and "breaks" in data_of["NGF"][site]:
            data_of[org][site]["breaks"] = data_of["NGF"][site]["breaks"]
    with breaks_update_output:
        display(f"Breaks added to {org} data")
    print(f"Breaks added to {org} data")

add_breaks_site_button.on_click(add_breaks_data_from_json)

def add_indv_breaks_data_from_NGF(_button):
    selected = list(ts_sites.value)
    clear_absolute_vector_render_cache()
    if not data_of.get("NGF"):
        with breaks_data_output:
            display(wg.HTML(
                '<em style="color:#B42318">The NGF catalog is the source of break '
                'data and is not loaded. Press "Latest NGF" first.</em>'))
        return
    if not selected:
        with breaks_data_output:
            display(wg.HTML(
                '<em style="color:#B42318">Select one or more stations in the '
                'Working Set first.</em>'))
        return
    for site in selected:
        siteid, org = remove_brac(site) # remove brac takes "P234 (UNR)" and returns ("P234", "UNR")
        if siteid in data_of["NGF"]:
            data_of[org][siteid]["breaks"] = data_of["NGF"][siteid]["breaks"]
    with breaks_data_output:
        display(f"NGF breaks data added to {siteid} ({org}) data")

add_breaks_ts_button.on_click(add_indv_breaks_data_from_NGF)

def remove_breaks_data_from_json(_button):
    org = breaks_select.value
    clear_absolute_vector_render_cache()
    if not data_of.get(org):
        with breaks_update_output:
            display(wg.HTML(
                '<em style="color:#B42318">' + org + ' catalog is not loaded, so it '
                'carries no breaks to remove.</em>'))
        return
    for site in data_of[org]:
        try:
            del data_of[org][site]["breaks"]
        except:
            pass
    with breaks_update_output:
        display(f"Breaks removed from {org} data")

remove_breaks_site_button.on_click(remove_breaks_data_from_json)

def remove_indv_breaks_data_from_NGF(_button):
    selected = list(ts_sites.value)
    clear_absolute_vector_render_cache()
    if not selected:
        with breaks_data_output:
            display(wg.HTML(
                '<em style="color:#B42318">Select one or more stations in the '
                'Working Set first.</em>'))
        return
    for site in selected:
        siteid, org = remove_brac(site) # remove brac takes "P234 (UNR)" and returns ("P234", "UNR")
        try:
            del data_of[org][siteid]["breaks"]
        except:
            pass
        display(f"NGF breaks data removed from {siteid} ({org}) data")
remove_breaks_ts_button.on_click(remove_indv_breaks_data_from_NGF)

#### Map

In [ ]:
## TODO: main map workflow ported to ipyleaflet; fixed bottom-left vector guide added
import html as html_module
# Map Functions
# Choices of maps are roughly the same as the old Folium version.

vector_guide = HTML()


def update_vector_guide(_change: object = None) -> None:
    """Update a compact, screen-fixed velocity-magnitude bracket.

    The bracket represents the Arrow reference magnitude in mm/yr; it is not
    a Google-style ground-distance scale bar. The separate annotation uses
    the live map-center latitude and zoom for a geographic equivalent.
    """
    if not hasattr(map[0], 'zoom'):
        return
    guide_vel = float(arrow_length_input.value)
    scale = float(velocity_scale_input.value)
    center_lat = float(map[0].center[0]) if map[0].center else 0.0
    metrics = velocity_guide_metrics(
        guide_vel, scale, center_lat, float(map[0].zoom), VELOCITY_REFERENCE_ZOOM,
        VELOCITY_REFERENCE_LATITUDE,
    )
    # The bracket width is the same reference-pixel conversion used by the
    # plotted arrows; do not add a visual clamp or arbitrary offset.
    guide_width_px = metrics["bar_px"]
    vector_guide.value = (
        '<style>.gnss-vector-div-icon,.gnss-vector-label-div-icon{pointer-events:none!important;background:transparent!important;border:0!important;}</style>'
        '<div class="gnss-velocity-guide" style="background:white; padding:6px 8px; border:1px solid #888; font-size:12px; line-height:1.2; width:max-content">'
        f'<div style="font-weight:600; margin-bottom:3px">{guide_vel:.2f} mm/yr vector magnitude</div>'
        f'<div style="position:relative; width:{guide_width_px:.1f}px; height:10px; margin:0 2px 3px 2px">'
        '<span style="position:absolute; left:0; right:0; top:4px; border-top:3px solid #111"></span>'
        '<span style="position:absolute; left:0; top:0; height:10px; border-left:2px solid #111"></span>'
        '<span style="position:absolute; right:0; top:0; height:10px; border-right:2px solid #111"></span></div>'
        f'<div style="color:#444">Geographic equivalent: {metrics["current_km"]:.2f} km at zoom {float(map[0].zoom):.0f}</div>'
        '</div>'
    )

def add_vector_guide(map_obj: Map) -> None:
    """Adds the fixed velocity guide to the lower-left corner of the map.

    Args:
        map_obj: ipyleaflet map object
    """
    # Put the guide on the map as a screen-fixed widget, not a geographic vector.
    update_vector_guide()
    map_obj.add(WidgetControl(widget=vector_guide, position='bottomleft'))
    map_obj.observe(update_vector_guide, names='zoom')
    map_obj.observe(update_vector_guide, names='center')


def BaseMap_Set(_button: object) -> object:
    """Selects the ipyleaflet basemap layer from the basemap widget value.

    Args:
        _button: button click event.

    Returns:
        The selected ipyleaflet basemap or TileLayer.
    """
    if BaseMap_butt.value == 'OpenTopo':
        return TileLayer(
            url="https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png",
            attribution=(
                'Map data: &copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> '
                'contributors, <a href="http://viewfinderpanoramas.org">SRTM</a> | '
                'Map style: &copy; <a href="https://opentopomap.org">OpenTopoMap</a>'
            ),
            name='OpenTopoMap'
        )
    elif BaseMap_butt.value == 'OpenStreetMap':
        return basemaps.OpenStreetMap.Mapnik
    else:
        return basemaps.Esri.WorldImagery


def ensure_live_dynamic_layers(map_obj: object) -> LayerGroup:
    """Resolve the visible dynamic group, never a detached staging group."""
    layers = tuple(getattr(map_obj, 'layers', ()))
    current = map_dynamic_layers[0]
    if current is not None and current in layers:
        return current
    for layer in layers:
        if isinstance(layer, LayerGroup) and getattr(layer, 'name', '') == 'Station and Velocity Layers':
            map_dynamic_layers[0] = layer
            return layer
    current = LayerGroup(name='Station and Velocity Layers')
    add_map_layer(map_obj, current)
    map_dynamic_layers[0] = current
    return current

def new_map(b):
    base_map = BaseMap_Set(b)

    ll = (40,-100) ; zoom = 4
    for option in list(map_radius_list.value):
        option = option.split(", ")
        ll = data_of[option[1]][option[0]]["location"]
        zoom = 6

    if hasattr(map[0], 'layers'):
        ensure_live_dynamic_layers(map[0]).clear_layers()
        map[0].panes = dict(GNSS_MAP_PANES)
        map[0].layout = Layout(width='100%', height='600px', margin='2% 0')
        map[0].basemap = base_map
        map[0].center = ll
        map[0].zoom = zoom
        update_vector_guide()
        return

    map[0] = Map(
        center=ll,
        zoom=zoom,
        basemap=base_map,
        scroll_wheel_zoom=True,
        panes=dict(GNSS_MAP_PANES),
        layout=Layout(width='100%', height='600px', margin='2% 0')
    )
    map_dynamic_layers[0] = LayerGroup(name='Station and Velocity Layers')
    map[0].add(map_dynamic_layers[0])
    map[0].add(WidgetControl(widget=map_loading, position='topright'))
    add_vector_guide(map[0])
    with map_output:
        display(map[0])
new_map_butt.on_click(new_map)

def reload_map(_button):
    """Builds replacement layers off-map, then reveals them together.

    Args:
        _button: reload button click event, or None for a programmatic refresh.
    """
    if not hasattr(map[0], 'layers') or map_reloading[0]:
        return

    map_reloading[0] = True
    reload_map_butt.disabled = True
    map_loading.layout.display = 'block'
    try:
        map[0].basemap = BaseMap_Set(_button)
        visible_layers = ensure_live_dynamic_layers(map[0])
        stage_layer_update(visible_layers, _render_map_selection)
    finally:
        map_loading.layout.display = 'none'
        reload_map_butt.disabled = False
        map_reloading[0] = False

reload_map_butt.on_click(reload_map)

def map_setting_changed(_change):
    """Refreshes visible map data after a discrete setting changes.

    Args:
        _change: widget change event for a map setting.
    """
    reload_map(None)

BaseMap_butt.observe(map_setting_changed, names='value')

def close_map(_button):
    map_output.clear_output()
    nearest_site_output.clear_output()
    graph_output.clear_output()
    map[0] = "Not initialized yet!"
    map_dynamic_layers[0] = None
close_map_butt.on_click(close_map)


####################
# SITE/RADIUS LIST #
####################

def scroll_new_map_station_into_view(expected_count: int) -> None:
    map_site_scroll_output.clear_output(wait=True)
    expected_count = max(1, int(expected_count))
    script = """
            (() => {
                const expectedCount = Math.max(1, Number.parseInt(__EXPECTED_COUNT__, 10) || 1);
                let attempts = 0;
                const sync = () => {
                    const viewports = [...document.querySelectorAll('.gnss-selected-stations-viewport')]
                        .filter(node => node.isConnected && node.offsetParent !== null);
                    const qualifying = viewports.filter(node => node.querySelectorAll('.gnss-selected-station-row').length >= expectedCount);
                    const viewport = qualifying[qualifying.length - 1];
                    if (viewport) { viewport.scrollTop = viewport.scrollHeight; return; }
                    if (++attempts < 12) requestAnimationFrame(sync);
                };
                requestAnimationFrame(sync);
            })();
    """
    with map_site_scroll_output:
        display(Javascript(script.replace('__EXPECTED_COUNT__', str(expected_count))))
def update_map_list(_button):
    siteid = site_id.value.strip().upper()
    radius = int(site_radius.value or 0)
    org = org_map_select.value
    if siteid in data_of.get(org, {}):
        addition = f"{siteid}, {org}, {radius}km"
        if addition in tuple(map_radius_list.options):
            return
        previous_count = len(map_radius_list.options)
        previous_selection = tuple(map_radius_list.value)
        first_site = not map_radius_list.options
        map_radius_list.options = list(map_radius_list.options) + [addition]
        if first_site:
            map_radius_list.value = (addition,)
        else:
            map_radius_list.value = tuple(
                option for option in previous_selection if option in map_radius_list.options
            )
        render_site_table()
        render_neighbor_table()
        if previous_count >= 5:
            scroll_new_map_station_into_view(len(map_radius_list.options))
    else:
        map_output.clear_output()
        with map_output:
            if not data_of.get(org):
                display(wg.HTML(
                    '<em style="color:#B42318">%s catalog is not loaded. Press '
                    '"Latest %s" in the Database Fetcher window, then try again.</em>'
                    % (org, org)))
            else:
                display(wg.HTML(
                    '<em style="color:#B42318">%s is not in the %s catalog.</em>'
                    % (siteid or '(blank)', org)))

    # MOD TAH 241223: Don't clear site name

add_sites_map_button.on_click(update_map_list)

def remove_map_list(_button):
    options = list(map_radius_list.options)
    for option in map_radius_list.value:
        options.remove(option)
    map_radius_list.options = options
    map_radius_list.value = tuple(option for option in map_radius_list.value if option in options)
    render_site_table()
    render_neighbor_table()

remove_sites_map_button.on_click(remove_map_list)

def selected_map_options_from_rows() -> tuple:
    """Return checked row options in the table's stable display order."""
    return tuple(
        option for option in map_radius_list.options
        if option in map_site_checks and map_site_checks[option].value
    )

def sync_map_site_selection(_change, option=None):
    """Copies selected table rows into the map's selected-site list.

    Args:
        _change: station-row selection change event.
    """
    # Bind each observer to its row option so a rebuilt/scrolled table cannot
    # accidentally read stale widgets from a previous render.
    if option is None:
        map_radius_list.value = selected_map_options_from_rows()
    else:
        selected = set(map_radius_list.value)
        if bool(_change.get('new')):
            selected.add(option)
        else:
            selected.discard(option)
        map_radius_list.value = tuple(
            candidate for candidate in map_radius_list.options if candidate in selected
        )
    render_neighbor_table()

def site_table_cell(widget, width, height='1.6rem'):
    """Places one widget inside a bordered station-table cell.

    Args:
        widget: widget to display in the cell.
        width: CSS width for the cell.

    Returns:
        A boxed widget with the station-table cell styling.
    """
    cell = wg.Box([widget], layout=wg.Layout(
        width=width, border='1px solid #b8b8b8', padding='2px 5px', min_width='0',
        height=height, min_height=height, max_height=height, overflow='hidden',
        justify_content='center', align_items='center'
    ))
    cell.add_class('gnss-table-cell')
    return cell

def table_header(text):
    """Builds one compact, non-wrapping header label for either table.

    Args:
        text: heading text to display.

    Returns:
        An HTML widget containing the formatted heading.
    """
    return wg.HTML(
        f'<b style="font-size: 0.80em; white-space: nowrap;">{text}</b>'
    )

def station_number_toggle(number, selected):
    """Builds a numbered station selector that greys itself when selected.

    Args:
        number: one-based row number to display.
        selected: whether the station row starts selected.

    Returns:
        A toggle button for selecting the station row.
    """
    selector = wg.ToggleButton(
        value=selected, description=str(number),
        layout=wg.Layout(width='100%', height='100%', padding='0')
    )

    def update_selector_style(change):
        selector.style.button_color = '#d9d9d9' if change['new'] else None

    update_selector_style({'new': selected})
    selector.observe(update_selector_style, names='value')
    return selector

def render_site_table():
    """Renders a fixed five-row viewport, scrolling only when more rows are stored."""
    map_site_checks.clear()
    # Percentage widths preserve the legacy labels while spanning the full center column.
    widths = ['7%', '30%', '25%', '38%']
    title_row = HBox([
    site_table_cell(
        wg.HTML('<div style="text-align:center;"><b>List of Selected Stations</b></div>'),
        '100%', height='2.2rem'
    )
], layout=wg.Layout(width='100%', min_width='0', flex='0 0 2.2rem', min_height='2.2rem', max_height='2.2rem'))
    header = HBox([
        site_table_cell(table_header('No.'), widths[0], height='2.0rem'),
        site_table_cell(table_header('Station ID'), widths[1], height='2.0rem'),
        site_table_cell(table_header('Source'), widths[2], height='2.0rem'),
        site_table_cell(table_header('Radius (km)'), widths[3], height='2.0rem'),
    ], layout=wg.Layout(width='100%', min_width='0', flex='0 0 2.0rem', min_height='2.0rem', max_height='2.0rem'))
    data_rows = []
    row_count = max(5, len(map_radius_list.options))
    for number in range(1, row_count + 1):
        if number <= len(map_radius_list.options):
            option = map_radius_list.options[number - 1]
            siteid, org, radius = option.split(', ')
            selector = station_number_toggle(number, option in map_radius_list.value)
            map_site_checks[option] = selector
            selector.observe(
                lambda change, option=option: sync_map_site_selection(change, option),
                names='value',
            )
        else:
            selector = wg.Label(str(number))
            siteid, org, radius = '', '', ''
        data_rows.append(HBox([
            site_table_cell(selector, widths[0]),
            site_table_cell(wg.Label(siteid), widths[1]),
            site_table_cell(wg.Label(org), widths[2]),
            site_table_cell(wg.Label(radius), widths[3]),
        ], layout=wg.Layout(width='100%', min_width='0', height='1.6rem', min_height='1.6rem', max_height='1.6rem', flex='0 0 1.6rem', overflow='hidden')))
    # Fixed title/header with a deterministic five-row data viewport.
    body_layout = wg.Layout(
        width='100%', min_width='0', height='8rem', min_height='8rem', max_height='8rem',
        overflow='hidden auto' if len(map_radius_list.options) > 5 else 'hidden',
    )
    data_body = VBox(data_rows, layout=body_layout)
    data_body.add_class('gnss-selected-stations-viewport')
    for row in data_rows:
        row.add_class('gnss-selected-station-row')
    site_table.children = (title_row, header, data_body, map_site_scroll_output)

def nearest_station_rows(org, siteid, count=10):
    """Returns the nearest other stations, excluding the selected station itself.

    Args:
        org: data source name.
        siteid: station code used as the reference location.
        count: maximum number of neighboring stations to return.

    Returns:
        Station records ordered from nearest to farthest.
    """
    candidates, _ = nearby_sites(org, siteid, float('inf'))
    others = [row for row in candidates if row[0] != siteid]
    return sorted(others, key=lambda row: row[3])[:count]

def render_neighbor_table():
    """Render nearest neighbors with absolute and differential NEU values."""
    widths = ['7%', '12%', '11%', '13%', '19%', '19%', '19%']
    headers = ['No.', 'Station', 'Source', 'Distance<br>(km)', 'Velocity<br>NEU', 'Sigma<br>NEU', 'Differential<br>NEU']
    neighbours = []
    if map_radius_list.value:
        siteid, org, _radius = map_radius_list.value[0].split(', ')
        reference_info = data_of.get(org, {}).get(siteid, {})
        for station, _lat, _lon, distance in nearest_station_rows(org, siteid):
            info = data_of.get(org, {}).get(station, {})
            neighbours.append(neighbor_velocity_row(station, org, distance, info, reference_info))
    def neu_markup(values):
        """Return escaped, stacked N/E/U components from a model list."""
        components = list(values) if values is not None else []
        labels = ('N', 'E', 'U')
        return ''.join(
            f'<span class="gnss-neighbor-neu-component">{label}: '
            f'{format_display_number(components[index] if index < len(components) else None)}</span>'
            for index, label in enumerate(labels)
        )

    body_rows = []
    for number in range(1, 11):
        model = neighbours[number - 1] if number <= len(neighbours) else None
        if model:
            station_value = model.get('station')
            source_value = model.get('source')
            station = html_module.escape(str(station_value), quote=True) if station_value else '—'
            source = html_module.escape(str(source_value), quote=True) if source_value else '—'
            distance = format_display_number(model.get('distance_km'))
            cells = [str(number), station, source, distance,
                     neu_markup(model.get('velocity')), neu_markup(model.get('sigma')),
                     neu_markup(model.get('differential'))]
        else:
            cells = [str(number), '—', '—', '—', neu_markup(None), neu_markup(None), neu_markup(None)]
        body_rows.append('<tr>' + ''.join(f'<td>{value}</td>' for value in cells) + '</tr>')
    colgroup = ''.join(f'<col style="width:{width}">' for width in widths)
    header_cells = ''.join(f'<th scope="col">{header}</th>' for header in headers)
    table_markup = (
        '<div class="gnss-neighbor-table-wrap">'
        '<style>'
        '.gnss-neighbor-table-wrap { width: 100%; min-height: 0; max-width: 100%; overflow: hidden; box-sizing: border-box; }'
        '.gnss-neighbor-table-wrap table { width: 100%; max-width: 100%; table-layout: fixed; border-collapse: collapse; font-size: clamp(0.58rem, 0.72vw, 0.82rem); }'
        '.gnss-neighbor-table-wrap caption { padding: 0.18em; text-align: center; font-weight: 700; line-height: 1; }'
        '.gnss-neighbor-table-wrap th, .gnss-neighbor-table-wrap td { border: 1px solid #b8b8b8; box-sizing: border-box; padding: 0.12em 0.16em; text-align: center; vertical-align: middle; word-break: normal; overflow-wrap: normal; line-height: 0.98; }'
        '.gnss-neighbor-table-wrap th { white-space: normal; line-height: 0.98; }'
        '.gnss-neighbor-table-wrap td { white-space: nowrap; overflow: hidden; }'
        '.gnss-neighbor-table-wrap .gnss-neighbor-neu-component { display: block; white-space: nowrap; line-height: 0.98; }'
        '</style><table aria-label="List of Nearest Neighbors (mm/yr)"><colgroup>'
        + colgroup + '</colgroup><caption>List of Nearest Neighbors (mm/yr)</caption>'
        + '<thead><tr>' + header_cells + '</tr></thead><tbody>'
        + ''.join(body_rows) + '</tbody></table></div>'
    )
    neighbor_table.children = (wg.HTML(table_markup, layout=wg.Layout(width='100%', min_width='0', flex='0 0 auto')), )

def parse_bulk_selection(text):
    """Parses individual row numbers and bracketed inclusive or exclusive ranges.

    Args:
        text: comma, space, or line-break delimited row numbers and ranges.
            Parentheses exclude an endpoint; square brackets include it.

    Returns:
        A set of one-based station-row numbers selected by the input.
    """
    return core_parse_bulk_selection(text)

def apply_bulk_selection(_button):
    """Selects listed station rows, then refreshes the sites and neighbors tables.

    Args:
        _button: Select button click event.
    """
    selected_numbers = parse_bulk_selection(bulk_selection_input.value)
    map_radius_list.value = tuple(
        option for number, option in enumerate(map_radius_list.options, start=1)
        if number in selected_numbers
    )
    render_site_table()
    render_neighbor_table()

def select_all_sites(_button):
    """Selects every station currently listed in the sites table.

    Args:
        _button: Select All button click event.
    """
    map_radius_list.value = tuple(map_radius_list.options)
    render_site_table()
    render_neighbor_table()

select_bulk_sites_butt.on_click(apply_bulk_selection)
select_all_sites_butt.on_click(select_all_sites)

################
# SITE BUTTONS #
################

def _render_map_selection(map_obj: object):
    """Draws selected stations once, keeping their markers above vector paths.

    Args:
        map_obj: map layer group that receives the rendered circles, vectors, and markers.
    """
    selected_options = parse_map_selection(map_radius_list.value)
    plotvec = bool(plot_vec_check.value)
    siglim = number_list(vel_siglim_form.value, [1, 1, 2])
    up_limit = colorscalefactor_input.value
    nearby_records = {}
    circle_centers = {}

    for (org, siteid), radius in selected_options.items():
        if radius > 0:
            nearby_records[(org, siteid)], circle_centers[(org, siteid)] = nearby_sites(org, siteid, radius)

    render_plan = build_map_render_plan(
        selected_options, nearby_records, plotvec, thin_velocities_input.value
    )
    for key in render_plan.circle_keys:
        basic_circle(map_obj, circle_centers[key], selected_options[key])
    for org, siteid in render_plan.vector_keys:
        plot_velocity(siteid, org, siglim, map_obj)

    # Add markers after vectors so their popup targets remain on top.
    for org, siteid in render_plan.marker_keys:
        value = get_vel(siteid, org)
        tooltip = (
            f"{siteid}, no vel. data"
            if not isinstance(value, tuple)
            else f"{siteid}, vel: {format_neu_vector(value[0])} mm/yr"
        )
        selected = (org, siteid) in selected_options
        plot_site_locations(
            map_obj, org, [data_of[org][siteid]["location"]], [siteid], [tooltip],
            color="black" if selected else "blue", dotrad=4 if selected else 3,
            popup_owner=resolve_popup_owner(map_obj, map_dynamic_layers[0]),
        )

    update_vector_guide()

def site_circle(b):
    # Reconcile the live row widgets before plotting; this also covers a
    # button click delivered immediately after a table rebuild.
    if map_site_checks:
        row_selection = selected_map_options_from_rows()
        if row_selection != tuple(map_radius_list.value):
            map_radius_list.value = row_selection
    if not hasattr(map[0], 'layers'):
        new_map(b)
    ensure_live_dynamic_layers(map[0])
    reload_map(b)

site_circle_submit.on_click(site_circle)


def nn_graph(_button):
    """Show the nearest-neighbor table and distance chart."""
    nearest_site_output.clear_output()
    graph_output.clear_output()
    for option in list(map_radius_list.value):
        siteid, org, radius_text = option.split(', ')
        radius = int(radius_text.removesuffix('km'))
        if radius <= 0:
            continue
        site_list, _ = nearby_sites(org, siteid, radius)
        neighbours = sorted((row for row in site_list if row[0] != siteid), key=lambda row: row[3])[:10]
        reference_info = data_of.get(org, {}).get(siteid, {})
        models = [neighbor_velocity_row(
            row[0], org, row[3], data_of.get(org, {}).get(row[0], {}), reference_info
        ) for row in neighbours]
        with nearest_site_output:
            display(wg.HTML('<h2>10 Nearest Neighbouring Sites:</h2>'))
            tableHTML = '<table><tr><th>Site</th><th>Distance</th><th>Velocity NEU (mm/yr)</th><th>Sigma NEU (mm/yr)</th><th>Differential NEU (mm/yr)</th></tr>'
            for model in models:
                tableHTML += (
                    f"<tr><td>{model['station']}</td><td>{format_display_number(model['distance_km'])} km</td>"
                    f"<td>{format_neu_vector(model['velocity'])}</td><td>{format_neu_vector(model['sigma'])}</td>"
                    f"<td>{format_neu_vector(model['differential'])}</td></tr>"
                )
            display(wg.HTML(tableHTML + '</table>'))
        if models:
            with graph_output:
                display(wg.HTML('<h2 style="text-align:center;">Distances to Nearby Sites (km)</h2>'))
                plt.bar([model['station'] for model in models], [float(model['distance_km']) for model in models])
                plt.show()

def get_vel(siteid: str, org: str) -> object:
    """Gets velocity and velocity uncertainty for a station.

    Args:
        siteid: 4-character station code
        org: data source name

    Returns:
        A tuple containing velocity and velocity uncertainty, or None if unavailable.
    """
    # jpl format {"AB01": {"location": [52.209501, -174.204758], "velocity": [-23.472, -7.111, 1.69], "velsig": [0.003, 0.002, 0.009]}
    try:
        return data_of[org][siteid]["velocity"], data_of[org][siteid]["velsig"]
    except:
        return None

def plot_velocity(siteid: str, org: str, siglim: list, map_obj: object = None) -> None:
    """Plots a station velocity vector if its uncertainties are below the limits.

    Args:
        siteid: 4-character station code
        org: data source name
        siglim: maximum allowed North, East, and Up velocity uncertainties
        map_obj: map or layer group where the vector is drawn
    """
    if get_vel(siteid, org): velocity, velsig = get_vel(siteid, org)
    else: return

    # Relative mode subtracts the selected reference station's horizontal velocity.
    relative_mode = velocity_mode_select.value == 'relative'
    if relative_mode:
        for option in list(map_radius_list.value):
            option = option.split(", ")
            site_ref = option[0]
            refvel = data_of[org][site_ref]["velocity"]

    else:
        refvel = [0,0,0]
    # Only update N and E values (leave height unchanged)
    pltvel = [velocity[0]-refvel[0], velocity[1]-refvel[1], velocity[2]]


    # -10 <=> rgb(0,0,255)
    # 0 <=> rgb(0,0,0)
    # 10 <=> rgb(255,0,0)
    # Get Up velocity scaling factor ±255/colorscalefactor
    # Convert from ±<Range> to colorfactor: Red down; Blue Up
    colorscalefactor = 255/colorscalefactor_input.value

    if all(velsig[i] <= siglim[i] for i in range(3)):
        zv = velocity[2]
        if zv > 0:
            rv, bv = 0, min(255, zv * colorscalefactor)
        else:
            rv, bv = min(255, zv * ( -colorscalefactor)), 0
        rv, bv = int(round(rv)), int(round(bv))
        color = f"#{rv:02x}00{bv:02x}"
        if map_obj is None:
            map_obj = map[0]
        draw_vector(
            map_obj, data_of[org][siteid]["location"], pltvel, velocity_scale_input.value, color=color,
            org=org, site=siteid, sigma=velsig, siglim=siglim,
            up_color_factor=colorscalefactor_input.value, relative=relative_mode,
        )


def draw_length(_button):
    length = arrow_length_input.value
    coords = length_coords_text.value.split(",")
    for i in range(len(coords)):
        coords[i] = float(coords[i])

    end_coords = calc_distance_earth(coords[0], coords[1], length)

    line = Polyline(locations=[coords, end_coords], weight=3)
    line.popup = HTML(value=f"{length} mm/yr")
    map[0].add(line)
    with map_output:
        map_output.clear_output()
        display(map[0])


def clear_map_list(_button):
    map_radius_list.options = []
    map_radius_list.value = ()
    render_site_table()
    render_neighbor_table()
clear_map_list_butt.on_click(clear_map_list)

# Update the fixed vector guide when the scale controls move.
arrow_length_input.observe(update_vector_guide, names='value')
velocity_scale_input.observe(update_vector_guide, names='value')


#### Timeseries

In [ ]:
# Note: This initialization was added to avoid a bug where the backend was not initialized before plotting, which caused errors in some environments.
backend_initialized = False

def update_ts_list(_button):
    siteid = ts_site_form.value.strip().upper()
    org_ts = org_ts_select.value
    catalog_ts = data_of.get(org_ts, {})

    if not catalog_ts:
        timeseries_output.clear_output()
        with timeseries_output:
            display(wg.HTML(
                '<em style="color:#B42318">' + org_ts + ' catalog is not loaded. '
                'Press "Latest ' + org_ts + '" in the Database Fetcher window, '
                'then try again.</em>'))
    elif siteid not in catalog_ts:
        timeseries_output.clear_output()
        with timeseries_output:
            display(wg.HTML('''<em style="color:red">Site not in ''' + org_ts + '''.</em>'''))
    else:
        extendedsiteid = siteid + " (" + org_ts_select.value + ")"
        if extendedsiteid not in ts_sites.options:
            ts_sites.options = list(ts_sites.options) + [extendedsiteid]


append_butt.on_click(update_ts_list)

def list_to_graph(_button: object) -> None:
    """Plots the currently selected time-series stations from the widget controls.

    Args:
        _button: button click event.
    """
    global tsfig, backend_initialized
    if not backend_initialized:
        backend_act(None)
    selected = list(ts_sites.value)
    timeseries_output.clear_output()

    if len(shift_dict_all) != 0:
        shift_dict_selected = {}
        for site in selected:
            if site in shift_dict_all:
                shift_dict_selected[site] = shift_dict_all[site]
    else:
        shift_dict_selected = {}

    if siglim_form.value:
        global SigLim
        nlim, elim, ulim = number_list(siglim_form.value, [10, 10, 30])
        SigLim = (nlim, elim, ulim)

    yearrange = [datetime(1,1,1), datetime(3000,1,1)] # [far past, far future]

    if selected:
        if start_year_form.value: yearrange[0] = datetime.strptime(start_year_form.value, "%Y-%m-%d")
        if end_year_form.value: yearrange[1] = datetime.strptime(end_year_form.value, "%Y-%m-%d")

        with timeseries_output:
            plot_ts_graph_list(
                selected,
                int(live_update_check.value),
                yearrange,
                breaks = remove_breaks_checkbox.value,
                detrend = detrend_check.value,
                errorbars = [error_bar_check.value, thin_error_bars.value],
                errorbar_outline = [error_bar_outline_check.value, error_bar_opacity.value],
                outlier = remove_outliers.value,
                resolution = plot_ts_res.value,
                shift = shift_dict_selected,
                )
    else:
        with timeseries_output:
            display(wg.HTML('''<em style="color:red">Select a set of sites to plot!</em>'''))


plot_ts_butt.on_click(list_to_graph)

def remove_site(_button):
    options = list(ts_sites.options)
    for site in ts_sites.value:
        options.remove(site)
        shift_dict_all.pop(site, None)
    ts_sites.options = options

remove_site_button.on_click(remove_site)

shift_dict_all = {}

def add_shift(_button):
    selected = list(ts_sites.value)
    shift_output.clear_output()
    if shift_value.value:
        for site in selected:
            shift_dict_all[site] = shift_value.value
    else:
        for site in selected:
            shift_dict_all.pop(site, None)

    with shift_output:
            print("shifts:")
            print(shift_dict_all)

update_customization.on_click(add_shift)

def display_break_data(_button):
    breaks_data_output.clear_output()

    for siteid in ts_sites.value:
        site, org = remove_brac(siteid)
        if "breaks" in data_of[org][site]:
            with breaks_data_output:
                display(wg.HTML(f"{siteid} Break Data"))
                tableHTML = f"<table><tr><th><b>Date (yr, m, d, hr, min)</b></th><th><b>Offset in mm (dN, sN, dE, sE, dU, sU)</b></th><th><b>Type</b></th><th><b>Description</b></th></tr>"
                for i in data_of[org][site]["breaks"]:
                    tableHTML += f'<tr><td>{i}</td><td>{data_of[org][site]["breaks"][i]["offsets"]}</td><td>{data_of[org][site]["breaks"][i]["type"]}</td><td>{data_of[org][site]["breaks"][i]["description"]}</td></tr>'
                display(wg.HTML(tableHTML + "</table>"))
        else:
            with breaks_data_output:
                display(f"{siteid}: No Breaks Data")

show_breaks_data_button.on_click(display_break_data)

def backend_act(_button: object) -> None:
    """Activates the selected matplotlib backend for time-series plotting.

    Args:
        _button: button click event.
    """
    global backend_initialized
    # Note: This seems to be needed to get interactive graphic to appear
    systype = sys.platform
    if( backend_butt.value == 'Inline') :
        try:
            plt.switch_backend('module://matplotlib_inline.backend_inline')
            get_ipython().run_line_magic('matplotlib', 'inline')
        except:
            with timeseries_output:
                print('Problem selecting inline')
    elif ( backend_butt.value == 'ipympl') :
        try:
            plt.close('all')
            get_ipython().run_line_magic('matplotlib', 'widget')
        except Exception as e:
            with timeseries_output:
                print('ipympl error:', e)
    else :
        print('Checking System',systype)
        if (systype=='darwin' ) :
            try:
                plt.switch_backend('macosx')
            except:
                with timeseries_output:
                    print('Problem with osx with ',systype)
        elif (systype == 'posix' or systype == 'linux'):
            try:
                plt.switch_backend('qt5agg')
            except:
                with timeseries_output:
                    print('Problem with qt5agg with ',systype)
        else :
            try:
                plt.switch_backend('qt')
            except:
                with timeseries_output:
                    print('Problem with qt with ',systype)

    backend_initialized = True
    with timeseries_output:
        print('Using backend ',mpl.get_backend(),systype)

backend_activate.on_click(backend_act)

def close_ts(_button):
    timeseries_output.clear_output()
close_ts_butt.on_click(close_ts)

def clear_list(_button):
    ts_sites.options = []
    shift_dict_all.clear()
clear_list_butt.on_click(clear_list)

def id_points(_button: object) -> None:
    """Lets the user click plotted points and prints offset rename commands.

    Args:
        _button: button click event.
    """
    global tsfig
    username = os.environ.get('USER', os.environ.get('USERNAME'))
    if ( backend_butt.value == 'Interactive') :
        pts = tsfig.ginput(n=10, show_clicks=True)
        cnt = 0
        for pt in pts :
            cnt += 1
            dt = pt[0]*86400
            epoch = datetime.fromtimestamp(dt,tz=timezone.utc).strftime("%Y %m %d %H %M")
            ptstr = f"Point {cnt:2d} Epoch {epoch} Value {pt[1]:.2f} mm"
            with timeseries_output:
                print(ptstr)

        cnt = 0
        for pt in pts :
            dt = pt[0]*86400
            epoch = datetime.fromtimestamp(dt,tz=timezone.utc).strftime("%Y %m %d %H %M")
            ax= tsfig.axes
            ax[2].axvline(x=pt[0], color='g', linestyle='dotted', linewidth=2)
            ax[1].axvline(x=pt[0], color='g', linestyle='dotted', linewidth=2)
            ax[0].axvline(x=pt[0], color='g', linestyle='dotted', linewidth=2)
            with timeseries_output:
                siteid = ts_sites.value[0][0:4]
                code = "_"+chr(ord('A') + cnt)+"PS"
                current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                print(" rename", siteid ,"   ",siteid+code,epoch,"                  ! GNSS_Analysis", username,current_time)
            cnt += 1
    else:
        with timeseries_output:
            display(wg.HTML('''<em style="color:red">ID only works with Interactive Backend activated'!</em>'''))

id_button.on_click(id_points)


### Windows Layout

In [ ]:
fetch_catalogue_meter = ui.meter()
fetch_breaks_meter = ui.meter()

def refresh_fetch_state(_event=None):
    """Restates which catalogs are loaded and what a break transfer would touch.

    Args:
        _event: ignored; accepts a button or a traitlets change dict.
    """
    # LOC is user-supplied, never downloaded, so it does not belong in a meter
    # about which remote catalogs have been fetched.
    rows = ui.catalogue_rows(data_of, [org for org in orglist if org != 'LOC'])
    ui.set_meter(fetch_catalogue_meter, ' &middot; '.join(
        '%s <b>%s</b> stn' % (org, format(count, ',d')) if count
        else '%s <span class="warn">not loaded</span>' % org
        for org, count, _breaks in rows))
    lookup = {org: (count, breaks) for org, count, breaks in rows}
    source_count, source_breaks = lookup.get('NGF', (0, 0))
    target = breaks_select.value
    target_count, target_breaks = lookup.get(target, (0, 0))
    warning = '' if source_breaks else ' &middot; <span class="warn">Add Breaks would copy nothing</span>'
    ui.set_meter(fetch_breaks_meter, 'NGF source <b>%s</b>/%s &middot; %s target <b>%s</b>/%s%s' % (
        format(source_breaks, ',d'), format(source_count, ',d'), target,
        format(target_breaks, ',d'), format(target_count, ',d'), warning))

def clear_fetch_log(_button=None):
    """Empties the download log.

    Args:
        _button: ignored click argument.
    """
    fetcher_output.clear_output()

data_root_meter = ui.meter()
gnss_browse_dir = [Path(DATA_DIR)]
gnss_browse_guard = [False]

def refresh_data_root_state(_event=None):
    """Redraws the current data directory, its meter, and the folder browser.

    Args:
        _event: ignored; accepts a button or a traitlets change dict.
    """
    data_root_label.value = (
        '<div class="gnss-kv" style="grid-template-columns:minmax(6rem,auto) minmax(0,1fr)">'
        '<dt>Current</dt><dd>%s</dd></div>' % html_module.escape(str(DATA_DIR)))
    local_count = len(data_of.get('LOC', {}))
    ui.set_meter(data_root_meter, '<b>%d</b> local file%s registered'
                 % (local_count, '' if local_count == 1 else 's'))

    here = gnss_browse_dir[0]
    entries = []
    if here.parent != here:
        entries.append('.. (up one level)')
    try:
        entries += sorted(child.name + '/' for child in here.iterdir()
                          if child.is_dir() and not child.name.startswith('.'))
    except PermissionError:
        entries.append('(permission denied)')
    # Setting .options fires the selection observer, which would navigate again;
    # the guard makes the programmatic rewrite distinguishable from a click.
    gnss_browse_guard[0] = True
    data_root_browser.options = entries
    data_root_browser.value = None
    gnss_browse_guard[0] = False
    data_root_input.value = str(here)

def on_browse_select(change):
    """Navigates into the clicked directory, or up one level.

    Args:
        change: traitlets change dict from the folder list.
    """
    if gnss_browse_guard[0] or not change.get('new'):
        return
    chosen = change['new']
    here = gnss_browse_dir[0]
    gnss_browse_dir[0] = here.parent if chosen.startswith('..') else here / chosen.rstrip('/')
    refresh_data_root_state()

def use_this_folder(_button=None):
    """Applies the typed or browsed directory as the data root.

    Args:
        _button: ignored click argument.
    """
    data_location_output.clear_output()
    with data_location_output:
        target = Path((data_root_input.value or '').strip() or gnss_browse_dir[0]).expanduser()
        if apply_data_root(target):
            gnss_browse_dir[0] = Path(DATA_DIR)
            print('Data directory set to', DATA_DIR)
            print('Loaded:', ', '.join('%s %d' % (o, len(data_of.get(o, {})))
                                       for o in orglist) or 'nothing yet')
        else:
            print('Cannot write to', target, '- pick another folder.')
    refresh_data_root_state()
    refresh_fetch_state()

def reset_data_root(_button=None):
    """Returns the data root to the folder beside the notebook.

    Args:
        _button: ignored click argument.
    """
    data_location_output.clear_output()
    with data_location_output:
        if apply_data_root(PROJECT_ROOT / 'data'):
            print('Data directory reset to', DATA_DIR)
        else:
            print('Cannot write to', PROJECT_ROOT / 'data')
    gnss_browse_dir[0] = Path(DATA_DIR)
    refresh_data_root_state()
    refresh_fetch_state()

def add_local_file(_button=None):
    """Registers a user-supplied CWU position file as a LOC station.

    The file is copied into DATA_DIR/LOC so the station survives a restart, and
    its metadata is derived on the spot so it behaves like a catalog station.

    Args:
        _button: ignored click argument.
    """
    data_location_output.clear_output()
    with data_location_output:
        raw = (local_file_input.value or '').strip()
        if not raw:
            print('Enter the path to a CWU-format .csv position file.')
            return
        source = Path(raw).expanduser()
        if not source.exists():
            print('File not found:', source)
            return
        site = source.name[:4].upper()
        try:
            record = get_local_cwu_metadata(source)
        except Exception as parse_error:
            print('Could not read', source.name, '-', parse_error)
            return
        make_if_absent(DATA_DIR / 'LOC')
        (DATA_DIR / 'LOC' / source.name).write_bytes(source.read_bytes())
        record['filename'] = source.name
        data_of.setdefault('LOC', {})[site] = record
        with open(DATA_DIR / 'LOC' / 'LOC_data.json', 'w') as handle:
            json.dump(data_of['LOC'], handle)
        print('Added', site, 'from', source.name)
        print('Velocity N,E,U (mm/yr):', ', '.join('%.2f' % v for v in record['velocity']))
    refresh_data_root_state()
    refresh_fetch_state()

fetch_window = VBox([
    ui.stylesheet(),
    ui.panel('Database Fetcher', [
        ui.section('Source Catalogs', [
            ui.action_grid([ui.primary(update_NGF_butt), ui.neutral(update_UNR_butt),
                            ui.neutral(update_JPL_butt)], columns=3),
        ], meter_widget=fetch_catalogue_meter),
        ui.log_panel(fetcher_output, 'Download Log', height='10rem',
                     actions=[ui.neutral(clear_fetch_log_butt, width='auto')]),
        ui.section('Data Location', [
            data_root_label,
            ui.field_row('Folder:', data_root_input),
            data_root_browser,
            ui.action_grid([ui.primary(data_root_use_butt),
                            ui.neutral(data_root_reset_butt)], columns=2),
            ui.field_row('Local file:', local_file_input),
            ui.action_grid([ui.neutral(add_local_file_butt)], columns=1),
            ui.log_well(data_location_output, height='7rem'),
        ], meter_widget=data_root_meter),
        ui.section('Bulk Break Transfer', [
            ui.field_row('Copy NGF breaks to:', breaks_select),
            ui.action_grid([ui.primary(add_breaks_site_button),
                            ui.destructive(remove_breaks_site_button)], columns=2),
        ], meter_widget=fetch_breaks_meter),
        ui.log_panel(breaks_update_output, 'Break Transfer Log', height='6rem'),
    ], subtitle='step 1 - refresh the local catalogs'),
# margin '0 auto' centres the whole panel, and with it every box inside, once
# max_width stops it filling a wide output area. Layout only - no control,
# handler, or binding is touched.
], layout=wg.Layout(width='100%', max_width=ui.GNSS_WINDOW_MAX, min_width='0',
                    margin='0 auto', align_items='center'))

ui.bind(data_root_browser, on_browse_select)
ui.bind_click(data_root_use_butt, use_this_folder)
ui.bind_click(data_root_reset_butt, reset_data_root)
ui.bind_click(add_local_file_butt, add_local_file)
refresh_data_root_state()

ui.bind(breaks_select, refresh_fetch_state)
ui.bind_click(clear_fetch_log_butt, clear_fetch_log)
for fetch_button in [update_NGF_butt, update_UNR_butt, update_JPL_butt,
                     add_breaks_site_button, remove_breaks_site_button]:
    ui.bind_click(fetch_button, refresh_fetch_state)
refresh_fetch_state()

avail_match_meter = ui.meter()

def refresh_avail_state(_event=None):
    """Reports how many codes match and renders the resolved station record.

    Args:
        _event: ignored; accepts a button or a traitlets change dict.
    """
    org = org_avail_select.value
    catalog = data_of.get(org) or {}
    count, sample, exact = ui.match_codes(catalog.keys(), site_searchbar.value)
    query = (site_searchbar.value or '').strip().upper()
    if not catalog:
        ui.set_meter(avail_match_meter, '<span class="warn">%s not loaded</span>' % org)
        availability_detail.value = ui.stat_grid_html(
            [], empty='Press "Latest %s" in the Database Fetcher first.' % org)
    elif not query:
        ui.set_meter(avail_match_meter, 'type to filter <b>%s</b> %s codes' % (format(count, ',d'), org))
        availability_detail.value = ui.stat_grid_html([], empty='Enter a station code above.')
    elif exact:
        ui.set_meter(avail_match_meter, '<span class="ok">exact match</span> in %s' % org)
        availability_detail.value = ui.stat_grid_html(
            ui.station_detail_pairs(catalog[query]), columns=1)
    elif count:
        ui.set_meter(avail_match_meter, '<b>%s</b> match(es) &middot; %s%s' % (
            format(count, ',d'), ', '.join(sample), '...' if count > len(sample) else ''))
        availability_detail.value = ui.stat_grid_html([], empty='No exact match yet.')
    else:
        ui.set_meter(avail_match_meter, '<span class="bad">0 matches</span> in %s (%s codes searched)'
                     % (org, format(len(catalog), ',d')))
        availability_detail.value = ui.stat_grid_html(
            [], empty='%s is not in %s.' % (query, org))

def commit_site_search(_event=None):
    """Runs the stored lookup and refreshes the Data View together.

    Args:
        _event: ignored; accepts a button or a traitlets change dict.
    """
    site_search(None)
    refresh_avail_state()

availability_window = VBox([
    ui.stylesheet(),
    ui.panel('Station Availability', [
        ui.section('Lookup', [
            ui.field_row('Site ID:', site_searchbar),
            ui.field_row('Source:', org_avail_select),
            ui.action_grid([ui.primary(site_search_submit), ui.neutral(clear_log_butt)], columns=2),
        ], meter_widget=avail_match_meter),
        ui.section('Data View', [availability_detail]),
        ui.log_panel(availability_output, 'Raw Record', height='11rem'),
    ], subtitle='step 2 - verify a station before you use it'),
], layout=wg.Layout(width='100%', max_width=ui.GNSS_WINDOW_MAX, min_width='0', margin='0'))

ui.commit_on_enter(site_searchbar, commit_site_search)
ui.bind(org_avail_select, refresh_avail_state)
ui.bind_click(site_search_submit, refresh_avail_state)
ui.bind_click(clear_log_butt, refresh_avail_state)
refresh_avail_state()
render_site_table()
render_neighbor_table()
left_content_width = '100%'
left_label_layout = wg.Layout(width='38%')
left_input_row_layout = wg.Layout(
    width=left_content_width, justify_content='space-between', align_items='center'
)
station_input_section = VBox([
    HBox([wg.Label('Station ID:', layout=left_label_layout), site_id], layout=left_input_row_layout),
    HBox([wg.Label('Radius (km):', layout=left_label_layout), site_radius], layout=left_input_row_layout),
    HBox([wg.Label('Source:', layout=left_label_layout), org_map_select], layout=left_input_row_layout),
], layout=wg.Layout(gap='3%'))

compact_action_layout = wg.Layout(width='145px', height='36px')
for action_button in [new_map_butt, reload_map_butt, close_map_butt]:
    action_button.layout = compact_action_layout

# Site-list actions stay with the station inputs they affect.
site_action_layout = wg.Layout(width='48.5%')
add_sites_map_button.layout = site_action_layout
remove_sites_map_button.layout = site_action_layout
clear_map_list_butt.layout = wg.Layout(width='100%')
left_action_grid = VBox([
    HBox([add_sites_map_button, remove_sites_map_button], layout=wg.Layout(
        width='100%', justify_content='space-between'
    )),
    clear_map_list_butt,
], layout=wg.Layout(width='100%', gap='3%'))
base_map_row = HBox([
    wg.Label('Base Map Choice:', layout=left_label_layout), BaseMap_butt,
], layout=left_input_row_layout)
site_circle_submit.layout = wg.Layout(width='100%')
def build_basemap_preview():
    """Creates a non-interactive preview using the current basemap choice.

    Returns:
        An ipyleaflet map widget with interaction controls disabled.
    """
    return Map(
        center=(40.0, -110.0), zoom=4, basemap=BaseMap_Set(None),
        zoom_control=False, attribution_control=False, dragging=False,
        double_click_zoom=False, scroll_wheel_zoom=False, touch_zoom=False,
        box_zoom=False, keyboard=False,
        layout=wg.Layout(width='100%', height='100%')
    )

base_map_preview = build_basemap_preview()
base_map_preview_container = wg.Box([base_map_preview], layout=wg.Layout(
    width='100%', flex='1 1 0', min_height='0', overflow='hidden'
))

def update_basemap_preview(_change):
    """Rebuilds the preview so every basemap source refreshes visibly.

    Args:
        _change: base-map selection change event.
    """
    global base_map_preview
    base_map_preview = build_basemap_preview()
    base_map_preview_container.children = (base_map_preview,)

BaseMap_butt.observe(update_basemap_preview, names='value')
base_map_section = VBox([
    base_map_row, site_circle_submit, base_map_preview_container,
], layout=wg.Layout(
    width='100%', flex='1 1 0', min_height='0', box_sizing='border-box',
    border='1px solid #d0d0d0', padding='3%', gap='3%'
))

vector_label_layout = wg.Layout(width='43%')
vector_field_layout = wg.Layout(width='55%')
vector_slider_layout = wg.Layout(width='34%')
vector_value_layout = wg.Layout(width='18%')
vector_row_layout = wg.Layout(width='100%', justify_content='space-between', align_items='center')

velocity_mode_select.layout = vector_field_layout
vel_siglim_form.description = ''
vel_siglim_form.layout = vector_field_layout
for vector_slider in [thin_velocities_input, arrow_length_input, velocity_scale_input, colorscalefactor_input]:
    vector_slider.description = ''
    vector_slider.readout = False
    vector_slider.layout = vector_slider_layout

decimate_value_input = wg.BoundedIntText(
    value=thin_velocities_input.value, min=thin_velocities_input.min, max=thin_velocities_input.max,
    step=thin_velocities_input.step, layout=vector_value_layout
)
arrow_length_value_input = wg.BoundedFloatText(
    value=arrow_length_input.value, min=arrow_length_input.min, max=arrow_length_input.max,
    step=arrow_length_input.step, layout=vector_value_layout
)
velocity_scale_value_input = wg.BoundedFloatText(
    value=velocity_scale_input.value, min=velocity_scale_input.min, max=velocity_scale_input.max,
    step=velocity_scale_input.step, layout=vector_value_layout
)
up_rate_value_input = wg.BoundedFloatText(
    value=colorscalefactor_input.value, min=colorscalefactor_input.min, max=colorscalefactor_input.max,
    step=colorscalefactor_input.step, layout=vector_value_layout
)
wg.jslink((thin_velocities_input, 'value'), (decimate_value_input, 'value'))
wg.jslink((arrow_length_input, 'value'), (arrow_length_value_input, 'value'))
wg.jslink((velocity_scale_input, 'value'), (velocity_scale_value_input, 'value'))
wg.jslink((colorscalefactor_input, 'value'), (up_rate_value_input, 'value'))

vector_type_row = HBox([
    wg.Label('Vector Type:', layout=vector_label_layout), velocity_mode_select,
], layout=vector_row_layout)
vector_limit_row = HBox([
    wg.Label('S.D. Limit (σ):', layout=vector_label_layout), vel_siglim_form,
], layout=vector_row_layout)
decimate_row = HBox([
    wg.Label('Decimate:', layout=vector_label_layout), thin_velocities_input, decimate_value_input,
], layout=vector_row_layout)
arrow_length_row = HBox([
    wg.Label('Guide (mm/yr):', layout=vector_label_layout), arrow_length_input, arrow_length_value_input,
], layout=vector_row_layout)
velocity_scale_row = HBox([
    wg.Label('Scale (km/mm/yr):', layout=vector_label_layout), velocity_scale_input, velocity_scale_value_input,
], layout=vector_row_layout)
up_rate_row = HBox([
    wg.Label('±U Rate (mm/yr):', layout=vector_label_layout), colorscalefactor_input, up_rate_value_input,
], layout=vector_row_layout)

site_configuration_section = VBox([
    station_input_section, left_action_grid,
], layout=wg.Layout(
    width='100%', flex='0 0 auto', box_sizing='border-box',
    border='1px solid #d0d0d0', padding='3%', gap='3%'
))

left_map_column = VBox(
    [site_configuration_section, base_map_section],
    layout=wg.Layout(
    width='29%', flex='0 0 29%', min_width='0', box_sizing='border-box',
    height='100%',
    justify_content='flex-start',
    align_items='stretch',
    gap='3%')
)

bulk_selection_row = HBox([
    bulk_selection_input, select_bulk_sites_butt,
], layout=wg.Layout(
    width='100%', min_width='0', height=bulk_selection_control_height,
    flex='0 0 auto', margin='0 0 0.2rem 0', justify_content='space-between', overflow='visible'
))
bulk_selection_section = VBox([
    bulk_selection_row, select_all_sites_butt,
], layout=wg.Layout(
    width=table_content_width, min_width='0', height='4.2rem', min_height='4.2rem',
    flex='0 0 auto', overflow='visible'
))
center_map_column = VBox(
    [site_table, bulk_selection_section, neighbor_table],
    layout=wg.Layout(
        width='42%', flex='0 0 42%', min_width='0', height='100%',
        box_sizing='border-box', border='1px solid #d0d0d0',
        justify_content='space-between',
        align_items='center')
)

plot_vec_check.layout = wg.Layout(width='100%')

vector_preview = HTML(layout=wg.Layout(width='100%', flex='1 1 0', min_height='0'))

def update_vector_preview(_change=None):
    """Shows the selected vector style without drawing a map layer.

    Args:
        _change: optional vector-control change event.
    """
    siglim = number_list(vel_siglim_form.value, [1, 1, 2])
    up_limit = colorscalefactor_input.value
    tick_markup = ''
    for tick_x, tick_value in zip(np.linspace(60, 306, 11), np.linspace(-up_limit, up_limit, 11)):
        tick_label = '0' if abs(tick_value) < 1e-9 else f'{tick_value:+g}'
        tick_markup += (
            f'<line x1="{tick_x:g}" y1="234" x2="{tick_x:g}" y2="256" stroke="#333" stroke-width="1"/>'
            f'<text x="{tick_x:g}" y="230" text-anchor="middle" fill="#333" font-size="8">{tick_label}</text>'
        )
    vector_preview.value = (
        '<div style="width:100%; height:100%; display:flex; flex-direction:column; '
        'border:1px solid #d0d0d0; padding:3%; box-sizing:border-box; text-align:center; background:#fafafa">'
        '<svg viewBox="0 0 320 270" width="100%" style="flex:1 1 auto; min-height:0" preserveAspectRatio="xMidYMid meet" aria-label="East north up velocity-vector reference">'
        '<defs><linearGradient id="up-rate-gradient" x1="0%" x2="100%">'
        '<stop offset="0%" stop-color="#ff0000"/><stop offset="50%" stop-color="#000000"/>'
        '<stop offset="100%" stop-color="#0000ff"/></linearGradient></defs>'
        '<line x1="70" y1="160" x2="240" y2="160" stroke="#707070" stroke-width="3" stroke-dasharray="8 7"/>'
        '<line x1="240" y1="160" x2="240" y2="35" stroke="#707070" stroke-width="3" stroke-dasharray="8 7"/>'
        '<circle cx="70" cy="160" r="15" fill="#1677ff" stroke="#000000" stroke-width="2"/>'
        '<line x1="70" y1="160" x2="240" y2="35" stroke="#000000" stroke-width="5" stroke-linecap="round"/>'
        '<polyline points="211,43 240,35 223,66" fill="none" stroke="#000000" stroke-width="5" stroke-linecap="round" stroke-linejoin="round"/>'
        '<text x="248" y="98" dominant-baseline="middle" fill="#000000" font-size="11" font-weight="600">Vₙ = y ± σₙ</text>'
        '<text x="155" y="190" text-anchor="middle" fill="#000000" font-size="13" font-weight="600">Vₑ = x ± σₑ</text>'
        '<circle cx="24" cy="244" r="10" fill="none" stroke="#000000" stroke-width="2"/>'
        '<circle cx="24" cy="244" r="3" fill="#000000"/>'
        '<text x="24" y="266" text-anchor="middle" textLength="48" lengthAdjust="spacingAndGlyphs" fill="#000000" font-size="10" font-weight="600">Vᵤ = z ± σᵤ</text>'
        + tick_markup +
        '<rect x="60" y="239" width="246" height="13" fill="url(#up-rate-gradient)" stroke="#777" stroke-width="1"/>'
        '<text x="183" y="268" text-anchor="middle" fill="#333" font-size="9" font-weight="600">Vᵤ reference (mm/yr)</text>'
        '</svg>'
        '</div>'
    )

update_vector_preview()
velocity_mode_select.observe(update_vector_preview, names='value')
colorscalefactor_input.observe(update_vector_preview, names='value')
vel_siglim_form.observe(update_vector_preview, names='value')

vector_preview_section = VBox([
    vector_type_row, vector_limit_row, decimate_row, arrow_length_row,
    velocity_scale_row, up_rate_row,
    vector_preview,
], layout=wg.Layout(
    width='100%', flex='1 1 0', min_height='0', gap='3%',
    justify_content='flex-start', align_items='center'
))

vector_settings_section = VBox([
    vector_preview_section,
    HBox([plot_vec_check],
        layout=wg.Layout(
        width='100%',
        justify_content='center',
        gap='8px',
    )),
], layout=wg.Layout(
    width='29%', flex='0 0 29%', min_width='0', height='100%',
    min_height='0', box_sizing='border-box',
    border='1px solid #c8c8c8', padding='1.5%', gap='2%',
    justify_content='flex-start', align_items='stretch'
))

station_selection_section = HBox(
    [left_map_column, center_map_column, vector_settings_section],
    layout=wg.Layout(
        width='100%',              # Fills the entire browser width
        height='clamp(610px, 34vw, 630px)',
        align_items = "stretch",           # Columns share the dashboard height
        justify_content='flex-start', # 29% / 42% / 29% dashboard columns
        gap='0',
        margin='0 0 24px 0'        # Adds 24px of buffer space below this entire row
    ))

# Global map actions apply to the whole canvas, not to one column.
global_map_actions = HBox(
    [new_map_butt, reload_map_butt, close_map_butt],
    layout=wg.Layout(justify_content='center')
)

table_cell_css = wg.HTML('''
<style>
.gnss-table-cell { font-size: 0.85em !important; }
.gnss-table-cell .widget-label,
.gnss-table-cell .widget-html-content,
.gnss-table-cell .widget-toggle-button { font-size: inherit !important; }
</style>
''')

map_window = VBox([
    table_cell_css,
    wg.HTML('<h1 style="text-align:center; margin:0 0 20px 0;">Map</h1>'),
    station_selection_section,
    global_map_actions,
    map_output,
    ])

ts_list_meter = ui.meter()
ts_source_meter = ui.meter()
ts_window_meter = ui.meter()
ts_sigma_meter = ui.meter()
ts_breaks_meter = ui.meter()
ts_shift_meter = ui.meter()
ts_display_meter = ui.meter()
ts_dpi_meter = ui.meter()
ts_backend_meter = ui.meter()

TS_DPI_BY_RESOLUTION = {'HD': 600, 'Regular': 300, 'Low Res': 100}

def refresh_ts_state(_event=None):
    """Rebuilds every timeseries readout, the preview, and the shared ribbon.

    Everything here is derived from widget state the notebook already holds, so
    no analysis result is invented and no numeric path is touched.

    Args:
        _event: ignored; accepts a button or a traitlets change dict.
    """
    total, chosen, per, per_selected, unresolved = ui.list_counts(
        ts_sites.options, ts_sites.value, orglist)
    matched, _sample, _exact = ui.match_codes(
        ts_sites.options, ts_filter_form.value, cap=0, contains=True)
    filter_text = (ts_filter_form.value or '').strip()
    extra = ' &middot; <span class="bad">%d unresolved</span>' % unresolved if unresolved else ''
    if filter_text:
        extra += ' &middot; <b>%d</b> match "%s"' % (matched, filter_text)
    ui.set_meter(ts_list_meter, '<b>%d</b> sites &middot; <b>%d</b> selected%s' % (total, chosen, extra))
    ui.set_meter(ts_source_meter, ' &middot; '.join(
        '%s <b>%d</b>/%d' % (org, per_selected[org], per[org]) for org in orglist))

    window_text, window_valid = ui.window_summary(start_year_form.value, end_year_form.value)
    ui.set_meter(ts_window_meter,
                 window_text if window_valid else '<span class="bad">%s</span>' % window_text)

    effective, sigma_valid, sigma_status = ui.sigma_state(siglim_form.value, SigLim)
    triple = '(%s)' % ', '.join(format(float(value), 'g') for value in effective)
    if sigma_status == 'inherited':
        ui.set_meter(ts_sigma_meter, 'in force <b>%s</b> mm' % triple)
    elif sigma_status == 'pending':
        ui.set_meter(ts_sigma_meter, 'in force <b>(%s)</b> mm &middot; pending <b>%s</b> mm on Plot'
                     % (', '.join(format(float(value), 'g') for value in SigLim), triple))
    else:
        ui.set_meter(ts_sigma_meter, '<span class="bad">unparseable</span> &middot; keeping <b>%s</b> mm' % triple)
    (remove_outliers.remove_class if detrend_check.value else remove_outliers.add_class)('gnss-gated')

    with_breaks, resolvable, selected_with, selected_epochs = ui.break_counts(
        ts_sites.options, ts_sites.value, data_of)
    breaks_text = 'catalogued <b>%d</b>/%d &middot; selection <b>%d</b> epochs across <b>%d</b>' % (
        with_breaks, resolvable, selected_epochs, selected_with)
    if remove_breaks_checkbox.value and chosen and not selected_with:
        breaks_text += ' &middot; <span class="warn">nothing to remove</span>'
    ui.set_meter(ts_breaks_meter, breaks_text)

    shifted = len(shift_dict_all)
    ui.set_meter(ts_shift_meter, '<b>%d</b> station%s offset' % (shifted, '' if shifted == 1 else 's'))

    display_parts = []
    if error_bar_check.value:
        display_parts.append('bars every <b>%d</b>' % thin_error_bars.value)
    if error_bar_outline_check.value:
        display_parts.append('envelope alpha <b>%.2f</b>' % error_bar_opacity.value)
    if remove_outliers.value is not None and not detrend_check.value:
        display_parts.append('<span class="warn">N sigma needs Detrend</span>')
    ui.set_meter(ts_display_meter, ' &middot; '.join(display_parts) if display_parts else 'plain lines')

    ui.set_meter(ts_dpi_meter, '<b>%d</b> dpi' % TS_DPI_BY_RESOLUTION[plot_ts_res.value])
    if backend_initialized:
        ui.set_meter(ts_backend_meter, 'active <b>%s</b>' % mpl.get_backend())
    else:
        ui.set_meter(ts_backend_meter, '<span class="warn">not activated</span>')

    ts_preview.value = ui.plot_preview_svg(
        error_bar_check.value, thin_error_bars.value, error_bar_outline_check.value,
        error_bar_opacity.value, not remove_breaks_checkbox.value, detrend_check.value,
        shift_value.value)

    shared_ribbon = ui.ribbon([
        ('Sites', '%d selected / %d' % (chosen, total)),
        ('Window', window_text),
        ('Sigma', triple + ' mm'),
        ('Detrend', 'on' if detrend_check.value else 'off'),
        ('Breaks', 'removed' if remove_breaks_checkbox.value else 'kept'),
        ('Outliers', 'off' if remove_outliers.value is None else '%d sigma' % remove_outliers.value),
        ('Backend', backend_butt.value),
        ('Figure', '%d dpi' % TS_DPI_BY_RESOLUTION[plot_ts_res.value]),
        ('Live fetch', 'on' if live_update_check.value else 'off'),
    ])
    ts_ribbon.value = shared_ribbon
    ts_results_ribbon.value = shared_ribbon

ts_data_column = ui.column('29%', [
    ui.section('Add Station', [
        ui.field_row('Site ID:', ts_site_form),
        ui.field_row('Source:', org_ts_select),
        ui.action_grid([ui.primary(append_butt)], columns=1),
    ]),
    ui.section('Working Set', [
        ui.field_row('Filter:', ts_filter_form),
        ts_sites,
        ui.action_grid([ui.neutral(remove_site_button),
                        ui.destructive(clear_list_butt)], columns=2),
    ], meter_widget=ts_list_meter, flex='1 1 auto'),
    ui.section('Acquisition', [
        ui.flag(live_update_check),
    ], meter_widget=ts_source_meter),
])

ts_analysis_column = ui.column('42%', [
    ui.section('Time Window', [
        ui.field_row('Start (YYYY-MM-DD):', start_year_form),
        ui.field_row('End (YYYY-MM-DD):', end_year_form),
    ], meter_widget=ts_window_meter),
    ui.section('Quality Filtering', [
        # 'global' is not decoration: list_to_graph assigns the parsed triple to
        # the module-level SigLim, which the Map reads too.
        ui.field_row('Sigma limit (N,E,U mm), global:', siglim_form),
        ui.field_row('Remove N sigma:', remove_outliers),
    ], meter_widget=ts_sigma_meter),
    ui.section('Trend and Breaks', [
        ui.flag(detrend_check),
        ui.flag(remove_breaks_checkbox),
        ui.action_grid([ui.neutral(show_breaks_data_button), ui.neutral(add_breaks_ts_button),
                        ui.destructive(remove_breaks_ts_button)], columns=3),
    ], meter_widget=ts_breaks_meter),
    ui.section('Offsets', [
        # plot_ts_graph does `ud = ud + shift*3`, so the Up component moves three
        # times the entered value. Stated in the label because a reader who
        # misses it misreads the plot.
        ui.slider_row('Shift N/E (mm), Up x3:', shift_slider, shift_value),
        ui.action_grid([ui.neutral(update_customization)], columns=1),
        ui.log_well(shift_output, height='5rem'),
    ], meter_widget=ts_shift_meter),
])

ts_display_column = ui.column('29%', [
    ui.section('Series Display', [
        ui.flag(error_bar_check),
        ui.slider_row('Every Nth bar:', thin_error_bars_slider, thin_error_bars),
        ui.flag(error_bar_outline_check),
        ui.slider_row('Envelope alpha:', error_bar_opacity_slider, error_bar_opacity),
    ], meter_widget=ts_display_meter),
    ui.section('Figure Output', [
        ui.field_row('Resolution:', plot_ts_res),
    ], meter_widget=ts_dpi_meter),
    ui.section('Backend', [
        ui.field_row('Backend:', backend_butt),
        ui.action_grid([ui.neutral(backend_activate), ui.neutral(id_button)], columns=2),
    ], meter_widget=ts_backend_meter),
    ui.section('Preview', [ts_preview], flex='1 1 auto'),
])

timeseries_window = VBox([
    ui.stylesheet(),
    ui.panel('Timeseries', [
        ts_ribbon,
        ui.dashboard([ts_data_column, ts_analysis_column, ts_display_column]),
        HBox([ui.primary(plot_ts_butt, width='14rem'), ui.neutral(close_ts_butt, width='10rem')],
             layout=wg.Layout(width='100%', min_width='0', justify_content='center', margin='0')),
    ], subtitle='select - window - filter - correct - style - plot'),
], layout=wg.Layout(width='100%', min_width='0', margin='0'))

timeseries_output_window = VBox([
    ui.stylesheet(),
    ui.panel('Timeseries Output', [
        ts_results_ribbon,
        ui.log_panel(breaks_data_output, 'Break Data', height='14rem'),
        ui.log_panel(timeseries_output, 'Velocity Fit and Figure', height=None),
    ], subtitle='break table - velocity fit - figure'),
], layout=wg.Layout(width='100%', min_width='0', margin='0'))

for ts_control in [ts_sites, ts_filter_form, start_year_form, end_year_form, siglim_form,
                   remove_outliers, detrend_check, remove_breaks_checkbox, error_bar_check,
                   error_bar_outline_check, thin_error_bars, error_bar_opacity, shift_value,
                   plot_ts_res, backend_butt, live_update_check]:
    ui.bind(ts_control, refresh_ts_state)
ui.bind(ts_sites, refresh_ts_state, names='options')
for ts_button in [append_butt, remove_site_button, clear_list_butt, update_customization,
                  add_breaks_ts_button, remove_breaks_ts_button, backend_activate, plot_ts_butt]:
    ui.bind_click(ts_button, refresh_ts_state)
for text_field in [start_year_form, end_year_form, siglim_form, ts_filter_form, ts_site_form]:
    text_field.continuous_update = False
refresh_ts_state()

# Retain-all-features gate: raises inside the notebook, so the end-to-end test
# fails loudly if a regroup ever drops a control out of a window.
ui.assert_reachable(fetch_window, {
    'update_NGF_butt': update_NGF_butt, 'update_UNR_butt': update_UNR_butt,
    'update_JPL_butt': update_JPL_butt, 'breaks_select': breaks_select,
    'add_breaks_site_button': add_breaks_site_button,
    'remove_breaks_site_button': remove_breaks_site_button,
    'clear_fetch_log_butt': clear_fetch_log_butt, 'fetcher_output': fetcher_output,
    'breaks_update_output': breaks_update_output,
    'data_root_label': data_root_label, 'data_root_input': data_root_input,
    'data_root_browser': data_root_browser, 'data_root_use_butt': data_root_use_butt,
    'data_root_reset_butt': data_root_reset_butt, 'local_file_input': local_file_input,
    'add_local_file_butt': add_local_file_butt,
    'data_location_output': data_location_output,
}, 'fetch_window')
ui.assert_reachable(availability_window, {
    'site_searchbar': site_searchbar, 'org_avail_select': org_avail_select,
    'site_search_submit': site_search_submit, 'clear_log_butt': clear_log_butt,
    'availability_detail': availability_detail, 'availability_output': availability_output,
}, 'availability_window')
ui.assert_reachable(timeseries_window, {
    'ts_site_form': ts_site_form, 'org_ts_select': org_ts_select, 'append_butt': append_butt,
    'ts_filter_form': ts_filter_form, 'ts_sites': ts_sites,
    'remove_site_button': remove_site_button, 'clear_list_butt': clear_list_butt,
    'live_update_check': live_update_check, 'start_year_form': start_year_form,
    'end_year_form': end_year_form, 'siglim_form': siglim_form,
    'remove_outliers': remove_outliers, 'detrend_check': detrend_check,
    'remove_breaks_checkbox': remove_breaks_checkbox,
    'show_breaks_data_button': show_breaks_data_button,
    'add_breaks_ts_button': add_breaks_ts_button,
    'remove_breaks_ts_button': remove_breaks_ts_button, 'shift_slider': shift_slider,
    'shift_value': shift_value, 'update_customization': update_customization,
    'shift_output': shift_output, 'error_bar_check': error_bar_check,
    'thin_error_bars_slider': thin_error_bars_slider, 'thin_error_bars': thin_error_bars,
    'error_bar_outline_check': error_bar_outline_check,
    'error_bar_opacity_slider': error_bar_opacity_slider,
    'error_bar_opacity': error_bar_opacity, 'plot_ts_res': plot_ts_res,
    'backend_butt': backend_butt, 'backend_activate': backend_activate, 'id_button': id_button,
    'ts_preview': ts_preview, 'ts_ribbon': ts_ribbon, 'plot_ts_butt': plot_ts_butt,
    'close_ts_butt': close_ts_butt,
}, 'timeseries_window')
ui.assert_reachable(timeseries_output_window, {
    'ts_results_ribbon': ts_results_ribbon, 'breaks_data_output': breaks_data_output,
    'timeseries_output': timeseries_output,
}, 'timeseries_output_window')


Tested on the following versions (as of 07/17/2026):

- Python: 3.11.15
- ipyleaflet: 0.20.0
- ipympl: 0.9.8
- earthscope_sdk: 1.5.0
- pandas: 3.0.2
- numpy: 2.4.6
- ipywidgets: 8.1.7
- matplotlib: 3.10.9
- geopy: 2.4.1
- requests: 2.34.2


## Interface and Downloads

To use the program for the first time, you must run the fetch_window display and click the "Latest" button for every source you want data for. This will download the data of all sites for the source (excluding time-series data) from the relevant website into the "data" folder. Upon using this program again, you may skip this step if you don't want to update the data from the web. The data, once downloaded, will show up in the data/"source" folder as "site"_data.json. 

## Availability

The "Check Availability" box allows you to if data exists for a site in a given source. It will directly output the data for that site as it is stored in the JSON file (in the form of a dictionary). 

Adding/Remove Breaks: If for all sites in the selected organization, if the site is also available in the NGF data, it will take the NGF break data and add it to the non-NGF version of the site, allowing for the "remove breaks" and "show breaks data" button to be available for the graphing portion of the notebook. A way to do this for individual sites can be found in the time-series section below. 



In [254]:
display(fetch_window, availability_window) # run to display fetcher and availability options

## Using The Map

Adding A Station: Inputting the combination of a valid site id, a desired marker observation radius, selecting a source of data origin, and then clicking "Add Site" will add the given station to the stations table. 

Stations Table: Here, select all stations you wish to interact with: To map the selected sites, click the "Plot Site(s)" button. This will show the sites as well as all other sites within the selected radii. To remove the selected sites from the table, click the "Remove Site(s)" button. To clear all sites from the stations table, click the "Clear All" button.

Distance Table: This table shows the distances of the 10 nearest neighboring sites to the center site. 
It also displays their velocities, velocity relative to the first site, and the velocity sigmas.  

Decimate: Thins out velocity vectors on the map. The input indicates how many "1 in x" vectors will be shown (i.e. if it's 10, then 1 in every 10 vectors will be displayed on the map).

Plot Velocities: Plots vectors showing station velocities. Absolute vectors show each station's individual velocity, while relative vectors show velocities relative to selected site(s).

Guide: Sets the velocity magnitude represented by the screen-fixed reference bracket, in mm/yr.

Scale: Controls the plotted-vector and guide length conversion. The separate geographic-equivalent readout changes with the live map zoom and center latitude.  

The color of the vectors shows vertical motions with <span style="color:red">reds indicating downward motion</span>, <span style="color:blue">blues indicating upwards motion</span>, and <span style="color:grey">blacks indicating near zero vertical motion</span>.  

±U Rate: Sets the range at which the "up velocity" color saturates

Base Map Choice: sets the base map with topography, street map, or ArcGIS imagery.



## Using the Timeseries
Site Id + Source Dropdown + Add to list: Input a Site ID with the source you want to add to the Site List. 

Live Update: Check to download the latest time-series data for selected sites, even if a previous version of the data already exists in the Data/ folder. 

Site List + Plot!: Select the Sites to be graphed in the Site List Box, and click "Plot!" to graph them. Selecting multiple sites will overlay their data on top of each other. 

Show Breaks Data: This button will show table(s) with all the breaks data available for the site(s), such as time, cause, type, and offsets. 

Remove Site: Removes the selected sites from the graph. 

Clear List + Close Graph: Self-explanatory.

Start/End: Set start and end dates for the data in the graph. Will only accept dates in the yyyy-mm-dd or yyyy-m-d format. 

SigLim: The limits on the standard deviations of the points of the time-series plot expressed in (north, east, up). Sigma must be less than the values of each component. 

Remove N σ: Only works if Detrend is also checked. Removes outlier data from the graphs. The number determines how strict the outlier determination is; all data outside the standard deviation multiplied by the Remove N σ number will be removed. The data is then detrended and run through the process again until the beginning and ending number of data points remain the same. The number of removed data points and the number of iterations will be given. 

Error Bars + Error Bar Outlines: Shows Error Bars and Error Bar outlines for the data. 

Error Bar Opacity + NErrBar: Changes the opacity of error bars and shows only every N'th errorbar. 

Shift Values + Update Shift: Shifts all the values of the selected sites up or down by the given mm. It is useful to align or separate two or more graph lines for better comparison. Click "Update Shift" to add the given shift value to the site(s). A dictionary displaying the offsets for each site is shown in the window  (if it is not shifted, it will not show up in the dictionary).  All sites can be selected to set the shift back to zero.

Copy/Clear Breaks Data: If the selected sites are also available in the NGF data, it will take the NGF break data and add it to the non-NGF version of the site, allowing for the "remove breaks" and "show breaks data" button to be available. Copying the NGF breaks can eb done for all sites in JPL/UNR time-series can be found in the availability section above. 

Backend Choice allows of type of time-series graphics.  Backend Activate <b>must</b> be used to activate the chosen backend.  
Errors activating the chosen BackEnd are displayed at the bottom of the time-series output window.  The ipympl Backend may need additional package installation.  This Backend allows the time-series plot to be Zoomed and Saved.  The Interactive Backend allows Zooming, Saving, and coordinates to be identified on the time-series plot.  When interactive is first selected, make sure to activate the Backend and check the error to see that it is activated without errors. 

On some systems, once the Backend is changed, the kernel needs to be restarted to change to a different backend.

ID points: Allows locations on the time-series plot to be identified only when the interactive backend is used.  The output includes the coordinates of the locations selected, and GLOBK site rename commands that can be used to add times of breaks in GLOBK/TSFIT solutions.  Multiple points can be selected with the middle mouse button/return ending selections, the right mouse button/delete removing points, and the left button/click selecting next point.  The graphics window needs to be active before the first point is selected. 

Detrended output example

| Site      |    Vn |   σVn |     Ve |   σVe |    Vu |   σVu |   WRMS N |   χn |   WRMS E |   χe |   WRMS U |   χu |   Num |
|:----------|------:|------:|-------:|------:|------:|------:|---------:|-----:|---------:|-----:|---------:|-----:|------:|
| P040-UNR  | -4.84 | 0.002 | -14.39 | 0.001 | -0.07 | 0.006 | 1.55     | 1.87 | 1.61     | 2.38 | 5.57     | 2.08 | 7146  |
| P040-JPL  | -4.86 | 0.002 | -14.41 | 0.001 | -0.01 | 0.005 | 1.66     | 2.24 | 1.54     | 2.54 | 5.41     | 2.26 | 7031  |
| P040-NGF | -4.96 | 0.004 | -14.34 | 0.003 | -0.32|  0.015 | 0.86     | 0.44 | 0.85     | 0.53 | 6.17     | 0.86 | 7262  |



In [255]:
display(map_window) # displays map and map settings

In [256]:
display(timeseries_window) # displays settings for graph

In [257]:
display(timeseries_output_window) # displays graph, breaks data, and shift tracker (tied to "Update Shift")